# CFPB Consumer Complaints — Bronze → Silver Transformation
### A fully documented walkthrough of the data-cleaning pipeline

| | |
|---|---|
| **Layer** | 🥉 Bronze (raw CSV) → 🥈 Silver (typed, standardized, validated Delta table) |
| **Source** | CFPB Consumer Complaint Database — `Consumer_Complaints.csv` |
| **Engine** | Microsoft Fabric · PySpark · Delta Lake |
| **Inputs / outputs** | `Files/bronze/consumer_complaints/Consumer_Complaints.csv` → `silver.consumer_complaints` · `silver.consumer_complaints_quarantine` · `Files/exports/consumer_complaints_full` |
| **Result** | **903,983** raw rows → **896,931** clean rows + **7,052** quarantined rows · 18 columns → 22 columns |

---

#### How to read this notebook

Every transformation step follows the same layout, so you always know where to look:

| Block | What you will find |
|---|---|
| 🏷️ **Column card** | A small table that labels the column: source name → silver name, data type, null count, operations applied |
| 🎯 **Purpose / 🧠 Logic** | *Why* the step exists and *how* it works, in plain English |
| 💻 **Code** | The PySpark code, commented line by line |
| 📊 **Output** | The result of the cell (captured from the original Microsoft Fabric run) |
| 💡 **What the output tells us** | A short interpretation of the numbers |

> **About the outputs.** Output tables are the results captured from the original Fabric execution of this pipeline. Long tables are shown in scrollable previews, and the free-text complaint narrative is omitted from previews. Where a finding was **independently re-computed against the supplied CSV** it is tagged 🔎 *re-verified*.
> **To re-run:** attach a Fabric lakehouse that contains the bronze CSV at the path above. The `spark` session and `display()` are provided by Fabric.

#### Contents
1. **Part A — Project overview** · summary, about the CFPB, medallion architecture, data dictionary, silver schema, roadmap, PySpark syntax guide
2. **Part B — Pipeline** · Steps 0–25, from ingestion to writing the Silver tables
3. **Part C — Lead reviewer's notes** · reconciliation, observations and recommendations

# Part A — Project overview
## A.1 Project summary

**Goal.** Turn the raw CFPB consumer-complaint export (*bronze*) into a trustworthy, analysis-ready table (*silver*) that can feed dashboards, trend analysis and machine-learning features downstream (*gold*).

**What the pipeline does**

| # | Stage | What happens |
|---|---|---|
| 1 | **Ingest** | Read the CSV with a parser configured for multi-line, quoted free text |
| 2 | **Profile** | Row/column counts, schema, uniqueness of `Complaint ID`, null profile of every column |
| 3 | **Type** | Convert the two date columns from text to real `date` values |
| 4 | **Validate** | Business rule — a complaint cannot be *sent to the company* before it was *received*. Failing rows go to a **quarantine** table, never silently dropped |
| 5 | **Standardize** | Map legacy category labels (product, sub-product, issue, sub-issue) to the current CFPB taxonomy; clean company names |
| 6 | **Repair** | Normalize state codes; rebuild ZIP codes that lost a leading zero; mask ZIPs to `NNNXX` |
| 7 | **Tidy** | Trim whitespace, give every column a consistent `snake_case` name, drop the heavy free-text narrative |
| 8 | **Publish** | Write Delta tables (`silver.consumer_complaints`, `silver.consumer_complaints_quarantine`) and a CSV export |

**Scale at a glance**

| Metric | Value |
|---|---|
| Raw rows (bronze) | 903,983 |
| Clean rows (silver) | 896,931 (99.22 %) |
| Quarantined rows | 7,052 (0.78 %) |
| Columns | 18 raw → 22 silver |
| Complaint dates | 2011-12-01 → 2017-11-15 |
| Distinct companies (raw spelling) | ≈ 4,500 |

## A.2 About the CFPB

The **Consumer Financial Protection Bureau (CFPB)** is a United States federal agency created by the *Dodd-Frank Wall Street Reform and Consumer Protection Act* (2010) in response to the 2008 financial crisis. It began operating in July 2011. Its mission is to make consumer financial markets — mortgages, credit cards, student loans, bank accounts, debt collection, credit reporting and similar products — work fairly for consumers.

**Why this dataset exists.** Consumers can submit complaints about financial products and services to the CFPB. The bureau forwards each complaint to the company, the company responds (typically within 15 days), and the consumer may review the response. The CFPB then publishes the complaints in its public **Consumer Complaint Database**, with personal information removed. Free-text *narratives* are published only when the consumer gives consent (the consent field first appears in this data on **2015-03-19**).

**How the data is used.** Regulators use it to spot emerging problems and company behaviour; researchers and journalists use it to study trends; companies use it to benchmark themselves.

> ⚠️ **Interpretation caveat.** The records are consumer-reported and the volume of complaints is *not* a direct measure of a company's quality — large companies receive more complaints simply because they have more customers. Treat counts as signals, not verdicts.

## A.3 Medallion architecture — where this notebook sits

```
  🥉 BRONZE                         🥈 SILVER                          🥇 GOLD
  raw, as received          →       cleaned, typed, validated    →     business aggregates
  Consumer_Complaints.csv           silver.consumer_complaints         (dashboards, KPIs, ML features)
  (never modified)                  silver.consumer_complaints_quarantine
        └──────────  THIS NOTEBOOK  ──────────┘
```

| Layer | Principle applied in this notebook |
|---|---|
| **Bronze** | Kept exactly as received — this notebook only *reads* it, so any step can be re-run or audited |
| **Silver** | One row per complaint (unique `complaint_id`), correct data types, consistent labels, documented rules |
| **Quarantine** | Rows that break a rule are *kept* with a reason (`dq_status`, `dq_rule`, `dq_reason`) so nothing disappears without a trace |

## A.4 Data dictionary — source (bronze) columns

18 columns, 903,983 rows. *Null %* is measured on the full bronze table (see Step 2.2).

| # | Source column | Bronze type | What it means | Typical values | Null % |
|--:|---|---|---|---|--:|
| 1 | `Date received` | string | Date the CFPB received the complaint | `3/12/2014` (M/d/yyyy text) | 0.00 |
| 2 | `Product` | string | Financial product the complaint is about | Mortgage, Debt collection, Credit reporting, Credit card, Student loan … (18 labels) | 0.00 |
| 3 | `Sub-product` | string | More specific type of the product | FHA mortgage, Checking account, Medical … | 26.01 |
| 4 | `Issue` | string | The main problem the consumer reported | Incorrect information on credit report, Loan modification … (166 labels) | 0.00 |
| 5 | `Sub-issue` | string | More detail on the issue | Account status, Debt is not mine … | 52.83 |
| 6 | `Consumer complaint narrative` | string | Consumer's own description of what happened (free text, personal data masked as `XXXX`) — published only with consent | Long text | 77.88 |
| 7 | `Company public response` | string | Optional, public-facing statement by the company | “Company has responded to the consumer and the CFPB and chooses not to provide a public response” | 71.46 |
| 8 | `Company` | string | Company the complaint is about | JPMORGAN CHASE & CO., EQUIFAX, INC. … (≈ 4,500 spellings) | 0.00 |
| 9 | `State` | string | US state / territory of the consumer | CA, TX, PR, AE … (62 codes) | 1.02 |
| 10 | `ZIP code` | string | Consumer ZIP code — full 5 digits or privacy-masked `NNNXX` | `48382`, `352XX` | 1.03 |
| 11 | `Tags` | string | Special-population flag | Older American · Servicemember · both | 86.06 |
| 12 | `Consumer consent provided?` | string | Did the consumer agree to publish the narrative? | Consent provided / not provided / withdrawn / Other | 58.47 |
| 13 | `Submitted via` | string | Channel used to file the complaint | Web, Referral, Phone, Postal mail, Fax, Email | 0.00 |
| 14 | `Date sent to company` | string | Date the CFPB forwarded the complaint to the company | `3/17/2014` (M/d/yyyy text) | 0.00 |
| 15 | `Company response to consumer` | string | How the company closed the complaint | Closed with explanation, Closed with monetary relief … (8 labels) | 0.00 |
| 16 | `Timely response?` | string | Did the company respond within the required time? | Yes / No | 0.00 |
| 17 | `Consumer disputed?` | string | Did the consumer dispute the company's response? | Yes / No — *not collected after 2017-04-22* | 14.98 |
| 18 | `Complaint ID` | integer | Unique identifier of the complaint | `759217` | 0.00 |

> 💡 **Why so many nulls?** Most are *structural*, not errors: the narrative, consent and public-response fields only exist for certain periods; `Tags` is blank for most consumers; and `Consumer disputed?` stopped being collected in April 2017. Steps 14–20 prove this with first/last-observed dates.

## A.5 Silver schema — what the output table looks like

22 columns. Seventeen source columns are kept (the narrative is dropped) and five are **derived**. Only three columns change data type; everything else stays `string`.

| # | Silver column | Silver type | Origin | Treatment |
|--:|---|---|---|---|
| 1 | `complaint_id` | int | `Complaint ID` | renamed |
| 2 | `date_received` | **date** | `Date received` | parsed from text, original dropped |
| 3 | `date_sent_to_company` | **date** | `Date sent to company` | parsed from text, original dropped |
| 4 | `product` | string | `Product` | trimmed |
| 5 | `product_standardized` | string | *derived* | legacy → current taxonomy |
| 6 | `sub_product` | string | `Sub-product` | renamed, trimmed |
| 7 | `sub_product_standardized` | string | *derived* | legacy → current taxonomy |
| 8 | `issue` | string | `Issue` | renamed |
| 9 | `issue_standardized` | string | *derived* | legacy → current wording |
| 10 | `sub_issue` | string | `Sub-issue` | renamed, trimmed |
| 11 | `sub_issue_standardized` | string | *derived* | legacy → current wording |
| 12 | `company` | string | `Company` | renamed |
| 13 | `company_clean` | string | *derived* | hidden characters / double spaces / curly quotes fixed |
| 14 | `state` | string | `State` | upper-cased, trimmed |
| 15 | `zip_code` | string | `ZIP code` | leading zeros restored, invalid → NULL, masked to `NNNXX` |
| 16 | `tags` | string | `Tags` | renamed, trimmed |
| 17 | `submitted_via` | string | `Submitted via` | renamed, trimmed |
| 18 | `consumer_consent_provided` | string | `Consumer consent provided?` | renamed, trimmed |
| 19 | `company_public_response` | string | `Company public response` | renamed, trimmed |
| 20 | `company_response_to_consumer` | string | `Company response to consumer` | renamed, trimmed |
| 21 | `timely_response` | string | `Timely response?` | renamed, trimmed |
| 22 | `consumer_disputed` | string | `Consumer disputed?` | renamed, trimmed |

Quarantine table (`silver.consumer_complaints_quarantine`) — 9 columns: `complaint_id`, `date_received`, `date_sent_to_company`, `product`, `company`, `state`, `dq_status`, `dq_rule`, `dq_reason`.

## A.6 Transformation roadmap

| Step | Column / topic | Operation | Rows affected |
|--:|---|---|---|
| 0 | Setup | Import PySpark functions | — |
| 1 | Ingest | Read bronze CSV, check size and schema | 903,983 read |
| 2 | Profile | `Complaint ID` uniqueness, null profile | — |
| 3 | `complaint_id` | Rename | all |
| 4–5 | Dates | Parse text → `date`, drop text originals | all |
| 6 | **Date-sequence rule** | Split into valid / quarantine | **7,052 quarantined** |
| 7–8 | Product, sub-product | Trim + standardize to current taxonomy | all valid |
| 9–10 | Issue, sub-issue | Standardize wording | all valid |
| 11 | Company | Remove hidden characters, collapse spaces, straighten apostrophes | 23 distinct names changed |
| 12 | State | Upper-case, trim, blank → NULL | all valid |
| 13 | ZIP | Repair leading zeros, null invalid values, mask to `NNNXX` | all valid |
| 14–20 | Tags, channel, consent, public response, company response, timely, disputed | Profile + trim + rename | all valid |
| 21 | `issue` | Rename | all valid |
| 22 | Narrative | Drop (77.9 % empty, heavy free text) | all valid |
| 23 | Validation | Unique IDs, final quality counters, column order | — |
| 24–25 | Publish | Delta silver table, quarantine table, CSV export | 896,931 + 7,052 |

## A.7 PySpark syntax guide — read this once, the rest becomes easy

| Syntax | What it does | Used for |
|---|---|---|
| `spark.read.option(k, v).csv(path)` | Reads a CSV into a distributed table (DataFrame), configured by options | Step 1 |
| `F.col("x")` | Refers to column `x` inside an expression. (`F` is the alias of `pyspark.sql.functions`) | everywhere |
| `df.withColumn("n", expr)` | **Adds** column `n` — or **replaces** it if it already exists | most steps |
| `df.withColumnRenamed("old", "new")` | Renames a column | renames |
| `df.drop("c")` | Removes a column | Steps 4, 5, 22 |
| `df.filter(cond)` | Keeps only rows where the condition is true | Step 6 |
| `F.when(c1, v1).when(c2, v2).otherwise(d)` | SQL `CASE WHEN` chain — conditions are checked **top to bottom, first match wins**; `otherwise` is the fallback | standardization |
| `.isin("a", "b")` | SQL `IN ('a','b')` | standardization |
| `&`  `\|`  `~` | AND · OR · NOT between conditions (each condition needs its own parentheses) | conditions |
| `F.trim(c)` / `F.upper(c)` | Remove leading/trailing spaces · upper-case | text cleaning |
| `c.rlike(regex)` | True if the text matches a regular expression | ZIP, state |
| `F.regexp_replace(c, pattern, repl)` | Replace text matching a pattern | company cleaning |
| `F.lpad(c, 5, "0")` | Left-pad text to 5 characters with zeros (`1001` → `01001`) | ZIP |
| `F.substring(c, 1, 3)` | Characters 1–3 (Spark counts from **1**) | ZIP masking |
| `F.to_date(c, "M/d/yyyy")` | Parse text such as `3/12/2014` into a real date | Steps 4–5 |
| `F.lit(x)` | A constant value column (`F.lit(None)` = NULL) | flags, nulling |
| `c.isNull()` / `c.isNotNull()` | NULL tests (never use `== None`) | profiling |
| `df.groupBy(c).count()` | Frequency table: how many rows per value | profiling |
| `df.agg(F.count, F.countDistinct, F.min, F.max)` | One-row summary statistics | profiling |
| `display(df)` | Fabric's interactive table viewer (shows up to 1,000 rows) | previews |
| `df.write.format("delta").mode("overwrite").saveAsTable(t)` | Saves a DataFrame as a managed Delta table | Step 24 |

> **Case-insensitive column names.** Spark resolves column names case-insensitively by default, so `F.col("product")` finds the column `Product`. The notebook relies on this in a few places and points it out where it matters.

---
# Part B — The pipeline

## Step 0 — Setup

**🎯 Purpose** Make PySpark's column functions available under the short alias `F`.
**🧠 Logic** `pyspark.sql.functions` holds every building block used below (`col`, `when`, `trim`, `to_date` …). Importing it as `F` keeps expressions short — `F.col("x")` instead of `pyspark.sql.functions.col("x")`. In Fabric the `spark` session and `display()` already exist, so nothing else is needed.

In [ ]:
# Import PySpark's SQL functions library under the alias "F".
# Every expression below uses it, e.g. F.col(), F.when(), F.trim(), F.to_date().
from pyspark.sql import functions as F

---
## Step 1 — Ingest the bronze file

| 🏷️ Column card | |
|---|---|
| **Object** | `df_bronze` — raw CSV loaded as a Spark DataFrame |
| **Source** | `Files/bronze/consumer_complaints/Consumer_Complaints.csv` |
| **Size** | 903,983 rows × 18 columns |

**🎯 Purpose** Load the raw file *without altering it* and confirm that nothing was lost during parsing.
**🧠 Logic** The file contains free-text complaint narratives that include commas, double quotes and line breaks. A plain `read.csv` would split such records into several broken rows, so the reader is configured explicitly — each option is explained in the code.

### 1.1 Read the CSV

In [ ]:
# Location of the raw file inside the lakehouse "Files" area (bronze layer = untouched source data).
bronze_path = 'Files/bronze/consumer_complaints/Consumer_Complaints.csv'

df_bronze = (
    spark.read                                   # start a DataFrame reader
    .option('header','true')                     # first line holds the column names, not data
    .option('inferschema','true')                # scan the data and guess types (Complaint ID -> integer, rest -> string)
    .option('multiline','true')                  # a single record may span several lines (narratives contain line breaks)
    .option('quote','"')                         # text containing commas is wrapped in double quotes
    .option('escape','"')                        # a quote inside text is written as two quotes ("")
    .option('unescapedQuoteHandling', 'STOP_AT_CLOSING_QUOTE')  # tolerate stray quotes: keep reading until the closing quote
    .option('nullValue','N/A')                   # the literal text N/A becomes NULL (empty fields also become NULL)
    .csv(bronze_path)                            # run the read on that file
)

# Fabric's interactive table viewer (first 1,000 rows).
display(df_bronze)

Date received,Product,Sub-product,Issue,Company,State,ZIP code,Submitted via,Complaint ID
3/12/2014,Mortgage,Other mortgage,"Loan modification,collection,foreclosure",M&T BANK CORPORATION,MI,48382,Referral,759217
10/1/2016,Credit reporting,null,Incorrect information on credit report,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL,352XX,Web,2141773
10/17/2016,Consumer Loan,Vehicle loan,Managing the loan or lease,"CITIZENS FINANCIAL GROUP, INC.",PA,177XX,Web,2163100
6/8/2014,Credit card,null,Bankruptcy,AMERICAN EXPRESS COMPANY,ID,83854,Web,885638
9/13/2014,Debt collection,Credit card,Communication tactics,"CITIBANK, N.A.",VA,23233,Web,1027760


> 💡 **What the output tells us** The file parsed into proper columns. Blank cells are shown as `null`, and rows such as the first one (Mortgage → *Loan modification, collection, foreclosure*) are one complete complaint each — the multi-line settings worked.

### 1.2 Confirm size

In [ ]:
# .count() triggers a Spark job and returns the number of rows.
print('Row Count',df_bronze.count())
# .columns is a plain Python list of column names; len() counts them.
print('Column Count',len(df_bronze.columns))

Row Count 903983
Column Count 18


> 💡 **What the output tells us** 903,983 complaints × 18 columns. This is the baseline every later step is reconciled against (Part C shows the final reconciliation: 903,983 = 896,931 + 7,052).

### 1.3 Inspect the schema (data types)

In [ ]:
# Prints the column tree: name, data type, and whether NULLs are allowed.
df_bronze.printSchema()

root
 |-- Date received: string (nullable = true)
 |-- Product: string (nullable = true)
 |-- Sub-product: string (nullable = true)
 |-- Issue: string (nullable = true)
 |-- Sub-issue: string (nullable = true)
 |-- Consumer complaint narrative: string (nullable = true)
 |-- Company public response: string (nullable = true)
 |-- Company: string (nullable = true)
 |-- State: string (nullable = true)
 |-- ZIP code: string (nullable = true)
 |-- Tags: string (nullable = true)
 |-- Consumer consent provided?: string (nullable = true)
 |-- Submitted via: string (nullable = true)
 |-- Date sent to company: string (nullable = true)
 |-- Company response to consumer: string (nullable = true)
 |-- Timely response?: string (nullable = true)
 |-- Consumer disputed?: string (nullable = true)
 |-- Complaint ID: integer (nullable = true)


> 💡 **What the output tells us**
> * Only `Complaint ID` was inferred as a number. Both date columns are still **text** (`M/d/yyyy` is not a format Spark auto-detects) — they must be converted explicitly (Steps 4–5).
> * Column names contain spaces, hyphens and `?` (`Sub-product`, `Timely response?`). These are awkward in code and SQL, so every column is renamed to `snake_case` along the way.

---
## Step 2 — Profile the bronze data

**🎯 Purpose** Before changing anything, measure the data: is the key unique, and where are the gaps? Every later cleaning decision is anchored on these numbers.

### 2.1 Is `Complaint ID` a reliable key?
**🧠 Logic** A primary key must be non-null and unique. Compare four counts in one pass: total rows, non-null IDs, distinct IDs, null IDs.

In [ ]:
complaint_id_check = df_bronze.agg(                       # .agg() = aggregate the whole table into one row
                            F.count('*').alias('Total Rows'),                 # count every row
                            F.count('Complaint ID').alias('Non_Null_Complaint_IDs'),   # count() on a column skips NULLs
                            F.countDistinct('Complaint ID').alias('Distinct_Complaint_IDs'),  # unique IDs only
                            F.sum(
                                F.when(F.col('Complaint ID').isNull(),1).otherwise(0)  # 1 for each NULL id, else 0 ...
                            ).alias('Null_Complaint_IDs')                              # ... summed = number of NULL ids
                        )

display(complaint_id_check)

Total Rows,Non_Null_Complaint_IDs,Distinct_Complaint_IDs,Null_Complaint_IDs
903983,903983,903983,0


> 💡 **What the output tells us** All three numbers equal 903,983 and there are **0 nulls** → `Complaint ID` is complete and unique. It is a safe primary key and is used to prove integrity again in Step 23.

### 2.2 Null profile of every column
**🧠 Logic** Loop over all 18 columns; for each, count NULLs and express them as a percentage of all rows. The result is collected into a small DataFrame.

In [ ]:
total_rows = df_bronze.count()                 # denominator for the percentages
null_summary = []                              # will collect one tuple per column

for column_name in df_bronze.columns:          # visit every column
    # rows where this column is NULL
    null_count = df_bronze.filter(F.col(column_name).isNull()).count()
    # rows in the column (always equals total_rows; kept as a cross-check)
    total_count = df_bronze.select(column_name).count()

    null_summary.append(
        (
            column_name,
            null_count,
            total_count,
            round((null_count/total_rows)*100,2)   # null percentage, 2 decimals
        )
    )
    # Build a DataFrame from the list so far (re-created each pass; only the final version is displayed).
    null_df = spark.createDataFrame(
    null_summary,
    ["Column_Name", "Null_Count", "Total_Count", "Null_Percentage"]
    ) 

display(null_df)

Column_Name,Null_Count,Total_Count,Null_Percentage
Date received,0,903983,0.00
Product,0,903983,0.00
Sub-product,235160,903983,26.01
Issue,0,903983,0.00
Sub-issue,477597,903983,52.83
Consumer complaint narrative,704013,903983,77.88
Company public response,646002,903983,71.46
Company,0,903983,0.00
State,9225,903983,1.02
ZIP code,9278,903983,1.03


> 💡 **What the output tells us**
> | Group | Columns | Reading |
> |---|---|---|
> | **Complete (0 %)** | `Date received`, `Product`, `Issue`, `Company`, `Submitted via`, `Date sent to company`, `Company response to consumer`, `Timely response?`, `Complaint ID` | Safe to use as-is |
> | **Optional detail** | `Sub-product` 26.0 %, `Sub-issue` 52.8 % | Not every product/issue has a deeper level — *structural* nulls |
> | **Period-dependent** | `Consumer consent provided?` 58.5 %, `Company public response` 71.5 %, `Consumer disputed?` 15.0 %, `Narrative` 77.9 % | Fields exist only for part of the timeline (proved in Steps 14–20) |
> | **Sparse flag** | `Tags` 86.1 % | Blank = no special flag |
> | **Genuine gaps** | `State` 1.02 %, `ZIP code` 1.03 % | Location not supplied by the consumer |
>
> Because no cleaning rule depends on a heavily-null column, **no rows need to be removed for nulls** — nulls are preserved as `NULL`.

> ⚙️ *Performance note:* this loop launches 36 Spark jobs (two per column). A single `df.select([F.sum(F.col(c).isNull().cast("int")) ...])` would give the same table in one pass.

---
## Step 3 — `complaint_id`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Complaint ID` → `complaint_id` |
| **Type** | integer → integer |
| **Nulls** | 0 (0.00 %) |
| **Operation** | rename |

**🎯 Purpose** Start the silver DataFrame and give the key a clean `snake_case` name.
**🧠 Logic** `df_silver = df_bronze` creates a *new reference* — Spark DataFrames are immutable, so every later `withColumn` returns a new DataFrame and bronze stays untouched.

In [ ]:
df_silver = df_bronze                                  # silver starts as a copy-by-reference of bronze (bronze is never modified)

df_silver = df_silver.withColumnRenamed(
    "Complaint ID",                                    # old name (has a space)
    "complaint_id"                                     # new snake_case name
)
display(df_silver)

complaint_id,Date received,Product,Company
759217,3/12/2014,Mortgage,M&T BANK CORPORATION
2141773,10/1/2016,Credit reporting,"TRANSUNION INTERMEDIATE HOLDINGS, INC."
2163100,10/17/2016,Consumer Loan,"CITIZENS FINANCIAL GROUP, INC."
885638,6/8/2014,Credit card,AMERICAN EXPRESS COMPANY
1027760,9/13/2014,Debt collection,"CITIBANK, N.A."


> 💡 **What the output tells us** The key column is now `complaint_id`; values are unchanged.

---
## Step 4 — `date_received`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Date received` → `date_received` |
| **Type** | string (`M/d/yyyy`) → **date** |
| **Nulls** | 0 (0.00 %) |
| **Operations** | profile · parse · drop original |

**🎯 Purpose** Dates stored as text cannot be sorted, compared or used in date arithmetic. Convert them to a true `date` type.
**🧠 Logic** Profile first, then parse with the pattern `M/d/yyyy` (`M` and `d` accept one *or* two digits, so both `3/9/2014` and `12/10/2015` work).

### 4.1 Profile the raw text

In [ ]:
date_received_check = df_bronze.select(
                        F.count("Date received").alias("Total_Rows"),                     # non-null values
                        F.sum(F.when(F.col("Date received").isNull(),1).otherwise(0)).alias("Null_Count"),  # NULLs
                        F.min("Date received").alias("Min_Value"),                         # smallest value ...
                        F.max("Date received").alias("Max_Value")                          # ... and largest
                    )
display(date_received_check)

Total_Rows,Null_Count,Min_Value,Max_Value
903983,0,1/1/2012,9/9/2017


> 💡 **What the output tells us** No missing dates. ⚠️ But `Min_Value = 1/1/2012` and `Max_Value = 9/9/2017` are **alphabetical** comparisons of *text*, not chronological ones (`"9/9/2017"` sorts after `"11/15/2017"`). The true range, visible once the column is a date (Steps 14–20), is **2011-12-01 → 2017-11-15** 🔎 *re-verified*. Min/max should be taken *after* parsing — see Part C.

### 4.2 Parse text → date

In [ ]:
df_silver = df_silver.withColumn(
             "date_received",                                   # new column
             F.to_date(F.col("Date received"), "M/d/yyyy")      # parse e.g. "3/12/2014" -> 2014-03-12
)

# Show the text and the parsed value side by side to confirm the conversion.
display(df_silver.select(
            "Date received",
            "date_received"
    )
)

Date received,date_received
3/12/2014,2014-03-12
10/1/2016,2016-10-01
10/17/2016,2016-10-17
6/8/2014,2014-06-08
9/13/2014,2014-09-13


> 💡 **What the output tells us** `3/12/2014` → `2014-03-12` — day and month are read in the right order (US format, month first).

### 4.3 Drop the text original

In [ ]:
# The text column is now redundant; keeping both would invite confusion about which one to use.
df_silver = df_silver.drop("Date received")
display(df_silver)

complaint_id,Product,Company
759217,Mortgage,M&T BANK CORPORATION
2141773,Credit reporting,"TRANSUNION INTERMEDIATE HOLDINGS, INC."
2163100,Consumer Loan,"CITIZENS FINANCIAL GROUP, INC."
885638,Credit card,AMERICAN EXPRESS COMPANY
1027760,Debt collection,"CITIBANK, N.A."


> 💡 **What the output tells us** The text column `Date received` is gone from the DataFrame; only the typed `date_received` is kept (it appears in the previews from Step 5.4 onward). *(The cached preview captured in the original run for this cell pre-dates the parse step — see Part C, note 12.)*

---
## Step 5 — `date_sent_to_company`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Date sent to company` → `date_sent_to_company` |
| **Type** | string (`M/d/yyyy`) → **date** |
| **Nulls** | 0 (0.00 %) |
| **Operations** | look · profile · parse · drop original |

**🎯 Purpose / 🧠 Logic** Same treatment as Step 4 — this is the date the CFPB forwarded the complaint to the company. Both dates must be real dates before they can be compared in the quality gate (Step 6).

### 5.1 Look at the raw text

In [ ]:
display(df_silver.select('Date sent to company'))      # raw text values

Date sent to company
3/17/2014
10/5/2016
10/20/2016
6/10/2014
9/13/2014
11/20/2013


> 💡 **What the output tells us** Same `M/d/yyyy` text format as the received date.

### 5.2 Profile the raw text

In [ ]:
date_sent_check = df_bronze.select(
                        F.count("Date sent to company").alias("Total_Rows"),
                        # NOTE: this null count is taken on "Date received" (see Part C, note 2) - harmless here because both columns have 0 nulls
                        F.sum(F.when(F.col("Date received").isNull(),1).otherwise(0)).alias("Null_Count"),
                        F.min("Date sent to company").alias("Min_Value"),
                        F.max("Date sent to company").alias("Max_Value")
                    )
display(date_sent_check)

Total_Rows,Null_Count,Min_Value,Max_Value
903983,0,1/1/2013,9/9/2017


> 💡 **What the output tells us** All 903,983 values present. As in 4.1, min/max are alphabetical text results (`1/1/2013` is *not* the earliest send date).

### 5.3 Parse text → date

In [ ]:
df_silver = df_silver.withColumn(
             "date_sent_to_company",
             F.to_date(F.col("Date sent to company"), "M/d/yyyy")
)

display(df_silver.select(
            "Date sent to company",
            "date_sent_to_company"
    )
)

Date sent to company,date_sent_to_company
3/17/2014,2014-03-17
10/5/2016,2016-10-05
10/20/2016,2016-10-20
6/10/2014,2014-06-10
9/13/2014,2014-09-13


> 💡 **What the output tells us** Parsing succeeded (`3/17/2014` → `2014-03-17`). Step 6 gives indirect proof that **no value failed to parse** — a failed parse would become `NULL`, and the valid and quarantined row counts add up to the full 903,983.

### 5.4 Drop the text original

In [ ]:
df_silver = df_silver.drop("Date sent to company")     # keep only the typed column
display(df_silver)

complaint_id,date_received,date_sent_to_company,Product
759217,2014-03-12,2014-03-17,Mortgage
2141773,2016-10-01,2016-10-05,Credit reporting
2163100,2016-10-17,2016-10-20,Consumer Loan
885638,2014-06-08,2014-06-10,Credit card
1027760,2014-09-13,2014-09-13,Debt collection


---
## Step 6 — Data-quality gate: date sequence

| 🏷️ Rule card | |
|---|---|
| **Rule name** | `DATE SEQUENCE` |
| **Rule** | `date_sent_to_company` must be **on or after** `date_received` |
| **Why** | The CFPB cannot forward a complaint before it received it — a violation signals a data-entry or time-stamp error |
| **Action** | Rows that fail → **quarantine** (kept, with a reason). Rows that pass → `df_silver_valid`, the table all later steps clean |

**🧠 Logic** The two filters are exact complements (`<` and `>=`), so every row lands in exactly one of the two tables — nothing can vanish.

### 6.1 How many rows break the rule?

In [ ]:
# Keep rows where the company was sent the complaint BEFORE the CFPB received it (impossible order).
date_sequence_check = (
    df_silver.filter(F.col('date_sent_to_company') < F.col('date_received'))
)

display(date_sequence_check.count())     # number of offending rows

7052

> 💡 **What the output tells us** **7,052 rows (0.78 %)** violate the rule.

### 6.2 Build the quarantine table

In [ ]:
df_quarantine = (
    df_silver.filter(
        F.col('date_sent_to_company') < F.col('date_received'))   # the failing rows
    .withColumn('dq_status',F.lit("FAIL"))                        # F.lit = constant value written on every row
    .withColumn('dq_rule',F.lit('DATE SEQUENCE'))                 # which rule failed
    .withColumn('dq_reason',F.lit('Date sent to company is earlier than date received')  # human-readable reason
)
)
display(df_quarantine)

complaint_id,date_received,date_sent_to_company,Product,Company,State,dq_status,dq_rule,dq_reason
730165,2014-02-25,2014-02-24,Debt collection,"Navient Solutions, LLC.",RI,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
210633,2012-12-15,2012-12-14,Credit card,BARCLAYS BANK DELAWARE,CA,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
722527,2014-02-20,2014-02-19,Mortgage,"Stearns Ventures, LLC",CA,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
663076,2014-01-10,2014-01-09,Mortgage,"BANK OF AMERICA, NATIONAL ASSOCIATION",MD,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
804419,2014-04-11,2014-04-10,Mortgage,OCWEN LOAN SERVICING LLC,FL,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received


> 💡 **What the output tells us** Each quarantined row keeps its identifiers plus three audit columns (`dq_status`, `dq_rule`, `dq_reason`). In every example the *sent* date is exactly **one day before** the *received* date.
>
> 🔎 *Re-verified on the full file:* all **7,052** failing rows are exactly **1 day** early, all fall between **2011-12-02 and 2014-04-26**, and none occur afterwards. That pattern points to a systematic date-stamping artefact in the early years rather than random typos (see Part C, note 3).

### 6.3 Keep the valid rows

In [ ]:
df_silver_valid = (
    df_silver.filter(F.col('date_sent_to_company') >= F.col('date_received'))   # complement of the rule above
)

display(df_silver_valid.count())      # rows that pass
display(df_silver_valid)              # preview

896931

complaint_id,date_received,date_sent_to_company,Product,Company,State
759217,2014-03-12,2014-03-17,Mortgage,M&T BANK CORPORATION,MI
2141773,2016-10-01,2016-10-05,Credit reporting,"TRANSUNION INTERMEDIATE HOLDINGS, INC.",AL
2163100,2016-10-17,2016-10-20,Consumer Loan,"CITIZENS FINANCIAL GROUP, INC.",PA
885638,2014-06-08,2014-06-10,Credit card,AMERICAN EXPRESS COMPANY,ID
1027760,2014-09-13,2014-09-13,Debt collection,"CITIBANK, N.A.",VA


> 💡 **What the output tells us** **896,931** rows pass. 896,931 + 7,052 = **903,983** → the split is lossless.
> From this point on, **all cleaning is applied to `df_silver_valid`**.

---
## Step 7 — `product` / `product_standardized`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Product` → `product` (trimmed) + `product_standardized` (derived) |
| **Type** | string → string |
| **Nulls** | 0 (0.00 %) · 18 distinct raw labels |
| **Operations** | rule-based standardization · trim |

**🎯 Purpose** In April 2017 the CFPB revised its product list (new names such as *Credit card or prepaid card* first appear on **2017-04-21/24**). The file therefore mixes **legacy** labels (2011–2017) and **current** labels. Trend analysis across years needs *one* label set.
**🧠 Logic** A `CASE WHEN` chain maps each legacy product (sometimes *depending on the sub-product*) to its current equivalent. Rules are checked top to bottom, **first match wins**. Anything not matched keeps its original value (`otherwise`).

| # | Legacy `Product` (+ sub-product) | → `product_standardized` |
|--:|---|---|
| 1 | Prepaid card + *Transit card / Other special purpose card*; Bank account + *Cashing a check without an account* | **unchanged** (no current equivalent) |
| 2 | Prepaid card + *Mobile wallet* | Money transfer, virtual currency, or money service |
| 3 | Consumer Loan + *Vehicle loan / lease* | Vehicle loan or lease |
| 4 | Consumer Loan + *Installment, Pawn, Personal line of credit, Title loan* | Payday loan, title loan, or personal loan |
| 5 | Other financial service + *Credit repair* | Credit reporting, credit repair services, or other personal consumer reports |
| 6 | Other financial service + *Check cashing, Debt settlement, FX, Money order, Refund anticipation check, Traveler's checks* | Money transfer, virtual currency, or money service |
| 7 | Credit card · Prepaid card | Credit card or prepaid card |
| 8 | Bank account or service | Checking or savings account |
| 9 | Credit reporting | Credit reporting, credit repair services, or other personal consumer reports |
| 10 | Payday loan | Payday loan, title loan, or personal loan |
| 11 | Money transfers | Money transfer, virtual currency, or money service |

### 7.1 Build `product_standardized`

In [ ]:
# "product" (lower-case) resolves to the column "Product": Spark column names are case-insensitive by default.
# The new column is built from the RAW product + trimmed sub-product, before "product" itself is trimmed in 7.2.
df_silver_valid = df_silver_valid.withColumn(
     "product_standardized",
    
  
    # Rule 1 - exceptions: keep the original product (no current-taxonomy equivalent)
    F.when(
        ((F.col("product") == "Prepaid card") &
         F.trim(F.col("Sub-product")).isin("Transit card", "Other special purpose card")) |
        ((F.col("product") == "Bank account or service") &
         (F.trim(F.col("Sub-product")) == "Cashing a check without an account")),
        F.col("product")
    )

     # Rule 2 - prepaid mobile wallets are money services
     .when(
        (F.col("product") == "Prepaid card") &
        (F.trim(F.col("Sub-product")) == "Mobile wallet"),
        "Money transfer, virtual currency, or money service"
    )

    # Rule 3 - vehicle loans / leases
    .when(
        (F.col("product") == "Consumer Loan" ) &
        F.trim(F.col("Sub-product")).isin("Vehicle loan", "Vehicle lease"),
        "Vehicle loan or lease"
    )
    # Rule 4 - small-dollar consumer loans
    .when(
        (F.col("product") == "Consumer Loan") &
        F.trim(F.col("Sub-product")).isin(
            "Installment loan", "Pawn loan", "Personal line of credit", "Title loan"),
        "Payday loan, title loan, or personal loan"
    )
    
    # Rule 5 - credit repair belongs with credit reporting
    .when(
        (F.col("product") == "Other financial service") &
        (F.trim(F.col("Sub-product")) == "Credit repair"),
        "Credit reporting, credit repair services, or other personal consumer reports"
    )

    # Rule 6 - misc. financial services -> money services
    # (note: none of the Traveler's-check spellings listed here equals the source value Traveler’s/Cashier’s checks - curly apostrophes, no spaces; see Part C, note 4)
    .when(
        (F.col("product") == "Other financial service") &
        F.trim(F.col("Sub-product")).isin(
            "Check cashing", "Debt settlement", "Foreign currency exchange",
            "Money order", "Refund anticipation check", "Traveler’s / cashier’s checks",
            "Traveler's/Cashier's checks", "Traveler's/Cashier's checks"),
        "Money transfer, virtual currency, or money service"
    )
    
    # Rule 7 - one combined card category
    .when(F.col("product").isin("Credit card", "Prepaid card"),
          "Credit card or prepaid card")
    # Rule 8 - renamed product
    .when(F.col("product") == "Bank account or service", "Checking or savings account")
    # Rule 9 - renamed product
    .when(F.col("product") == "Credit reporting",
          "Credit reporting, credit repair services, or other personal consumer reports")
    # Rule 10 - renamed product
    .when(F.col("product") == "Payday loan", "Payday loan, title loan, or personal loan")
    # Rule 11 - renamed product
    .when(F.col("product") == "Money transfers",
          "Money transfer, virtual currency, or money service")

    # Fallback - already-current labels (Mortgage, Student loan, Debt collection ...) pass through unchanged
    .otherwise(F.col("product"))

)

display(df_silver_valid)

Product,Sub-product,product_standardized
Credit reporting,null,"Credit reporting, credit repair services, or other personal consumer reports"
Consumer Loan,Vehicle loan,Vehicle loan or lease
Credit card,null,Credit card or prepaid card
Bank account or service,Checking account,Checking or savings account
Bank account or service,Other bank product/service,Checking or savings account
Other financial service,Credit repair,"Credit reporting, credit repair services, or other personal consumer reports"
Prepaid card,General purpose card,Credit card or prepaid card
Money transfers,Domestic (US) money transfer,"Money transfer, virtual currency, or money service"


> 💡 **What the output tells us** Legacy names are rewritten to the current taxonomy (e.g. *Credit reporting* → *Credit reporting, credit repair services, or other personal consumer reports*). The full before/after audit with row counts is in Step 8.4.

### 7.2 Trim `product`

In [ ]:
# Replace "Product" with a trimmed copy named "product" (lower-case name -> snake_case convention).
# F.trim('Product') accepts a column name as text.
df_silver_valid = df_silver_valid.withColumn('product',F.trim('Product'))

display(df_silver_valid)

product,product_standardized,complaint_id
Mortgage,Mortgage,759217
Credit reporting,"Credit reporting, credit repair services, or other personal consumer reports",2141773
Consumer Loan,Vehicle loan or lease,2163100
Credit card,Credit card or prepaid card,885638
Debt collection,Debt collection,1027760


> 💡 **What the output tells us** `product` is the trimmed original and `product_standardized` is the harmonized label. 🔎 *Re-verified:* no product value actually had stray spaces — trimming is a safety net. **Both columns are kept** so analysts can always trace a standardized value back to its source.

---
## Step 8 — `sub_product` / `sub_product_standardized`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Sub-product` → `sub_product` (trimmed) + `sub_product_standardized` (derived) |
| **Type** | string → string |
| **Nulls** | 235,160 in bronze (26.01 %) → 232,240 in the valid set (25.9 %) |
| **Operations** | rename · trim · standardize |

**🎯 Purpose** Same legacy-vs-current problem as products, one level deeper (e.g. *Other mortgage* → *Other type of mortgage*).
**🧠 Logic** Sub-product meanings depend on the parent product, so every rule tests **product + sub-product** together.

### 8.1 Rename and trim

In [ ]:
# Rename "Sub-product" (hyphen) to snake_case.
df_silver_valid = df_silver_valid.withColumnRenamed(
    "Sub-product",
    "sub_product"
)

# Remove leading/trailing spaces so identical labels are not split into two groups.
df_silver_valid = df_silver_valid.withColumn(
     "sub_product",
     F.trim(F.col("sub_product"))
)

display(df_silver_valid)

product,sub_product,complaint_id
Mortgage,Other mortgage,759217
Credit reporting,null,2141773
Consumer Loan,Vehicle loan,2163100
Credit card,null,885638
Debt collection,Credit card,1027760


### 8.2 Profile the labels

In [ ]:
# Frequency of every sub-product label, Z -> A.
display(df_silver_valid.groupBy("sub_product").count().orderBy(F.col('sub_product').desc()))

sub_product,count
Virtual currency,453
Vehicle loan,17677
Vehicle lease,2646
VA mortgage,5742
Traveler’s/Cashier’s checks,88
Traveler's check or cashier's check,63
Transit card,37
Title loan,754
Store credit card,2047
Second mortgage,662


> 💡 **What the output tells us** 75 distinct labels plus `NULL` (232,240 rows). **Legacy and current spellings coexist**, e.g. `Other mortgage` (86,476) next to `Other type of mortgage` (3,421); `Medical` (21,013) next to `Medical debt` (4,313); `Traveler’s/Cashier’s checks` (curly apostrophe) next to `Traveler's check or cashier's check`. These duplicates are what the next step merges.

### 8.3 Standardize

In [ ]:
# Rules test the RAW trimmed product (column "product"), not product_standardized.
df_silver_valid = df_silver_valid.withColumn(
     "sub_product_standardized",

# DEBT COLLECTION: "Credit card" -> "Credit card debt", "Medical" -> "Medical debt", "Other (i.e. phone...)" -> "Other debt" ...
F.when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Auto'),
        'Auto debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Credit card'),
        'Credit card debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Federal student loan'),
        'Federal student loan debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Medical'),
        'Medical debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Mortgage'),
        'Mortgage debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Non-federal student loan'),
        'Private student loan debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Other', 'Other (i.e. phone, health club, etc.)'),
        'Other debt'
    )
    .when(
        (F.col("product") == 'Debt collection') &
        F.col("sub_product").isin('Payday loan'),
        'Payday loan debt'
    )

    # MORTGAGE: fixed + adjustable -> "Conventional home mortgage"; "Other mortgage" -> "Other type of mortgage" ...
    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Conventional adjustable mortgage (ARM)', 'Conventional fixed mortgage'),
        'Conventional home mortgage'
    )
    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Home equity loan or line of credit'),
        'Home equity loan or line of credit (HELOC)'
    )
    .when(
        (F.col("product") == 'Mortgage') &
        F.col("sub_product").isin('Other', 'Other mortgage'),
        'Other type of mortgage'
    )
    
    # PREPAID CARD: legacy card names -> current names (Gift card, Government benefit card ...)
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('General purpose card'),
        'General-purpose prepaid card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Gift or merchant card'),
        'Gift card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Electronic Benefit Transfer / EBT card', 'Government benefit payment card'),
        'Government benefit card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('ID prepaid card'),
        'Student prepaid card'
    )
    .when(
        (F.col("product") == 'Prepaid card') &
        F.col("sub_product").isin('Mobile wallet'),
        'Mobile or digital wallet'
    )

    # BANK ACCOUNT: "(CD) Certificate of deposit" -> "CD (Certificate of Deposit)"; "Other bank product/service" -> "Other banking product or service"
    .when(
        (F.col("product") == 'Bank account or service') &
        F.col("sub_product").isin('(CD) Certificate of deposit'),
        'CD (Certificate of Deposit)'
    )
    .when(
        (F.col("product") == 'Bank account or service') &
        F.col("sub_product").isin('Other bank product / service', 'Other bank product/service'),
        'Other banking product or service'
    )

    # CONSUMER LOAN (lower-case "loan"): never matches because the product is spelled "Consumer Loan" - harmless duplicate, see Part C note 6
    .when(
        (F.col("product") == 'Consumer loan') &
        F.col("sub_product").isin('Vehicle loan'),
        'Loan'
    )
    .when(
        (F.col("product") == 'Consumer loan') &
        F.col("sub_product").isin('Vehicle lease'),
        'Lease'
    )

    # STUDENT LOAN: "Non-federal student loan" -> "Private student loan"
    .when(
        (F.col("product") == 'Student loan') &
        F.col("sub_product").isin('Non-federal student loan'),
        'Private student loan'
    )
    
    # OTHER FINANCIAL SERVICE: credit repair / check cashing / traveler's checks
    # (the regexp_replace turns curly apostrophes into straight ones before comparing)
    .when(
        (F.col("product") == "Other financial service") &
        (F.col("sub_product") == "Credit repair"),
        "Credit repair services"
    )
    .when(
        (F.col("product") == "Other financial service") &
        (F.col("sub_product") == "Check cashing"),
        "Check cashing service"
    )
    .when(
        (F.col("product") == "Other financial service") &
        F.regexp_replace(F.col("sub_product"), "\u2019", "'").isin(
            "Traveler's / cashier's checks",
            "Traveler's/Cashier's checks"
        ),
        "Traveler's check or cashier's check"
    )

    # CONSUMER LOAN: vehicle loan -> "Loan", vehicle lease -> "Lease"
    .when(
        (F.col("product") == 'Consumer Loan') &
        F.col("sub_product").isin('Vehicle loan'),
        'Loan'
    )
    .when(
        (F.col("product") == 'Consumer Loan') &
        F.col("sub_product").isin('Vehicle lease'),
        'Lease'
    )

    # Fallback - keep the existing label (this includes NULL sub-products)
    .otherwise(F.col("sub_product"))
)

display(df_silver_valid)

product,sub_product,sub_product_standardized
Mortgage,Other mortgage,Other type of mortgage
Consumer Loan,Vehicle loan,Loan
Debt collection,Credit card,Credit card debt
Mortgage,Conventional adjustable mortgage (ARM),Conventional home mortgage
Debt collection,Medical,Medical debt
Mortgage,Conventional fixed mortgage,Conventional home mortgage
Student loan,Non-federal student loan,Private student loan
Debt collection,"Other (i.e. phone, health club, etc.)",Other debt


### 8.4 Audit the mapping
**🧠 Logic** Group by all four columns to see exactly what became what, and how many rows each mapping touched.

In [ ]:
display(df_silver_valid.groupBy(
    "product", "sub_product", "product_standardized", "sub_product_standardized"
).count().orderBy("product", "sub_product"))

product,sub_product,product_standardized,sub_product_standardized,count
Bank account or service,(CD) Certificate of deposit,Checking or savings account,CD (Certificate of Deposit),3405
Bank account or service,Cashing a check without an account,Bank account or service,Cashing a check without an account,619
Bank account or service,Checking account,Checking or savings account,Checking account,58748
Bank account or service,Other bank product/service,Checking or savings account,Other banking product or service,17765
Bank account or service,Savings account,Checking or savings account,Savings account,5270
Checking or savings account,CD (Certificate of Deposit),Checking or savings account,CD (Certificate of Deposit),268
Checking or savings account,Checking account,Checking or savings account,Checking account,7454
Checking or savings account,Other banking product or service,Checking or savings account,Other banking product or service,1639
Checking or savings account,Personal line of credit,Checking or savings account,Personal line of credit,7
Checking or savings account,Savings account,Checking or savings account,Savings account,579


> 💡 **What the output tells us**
> * **Merges work:** `Conventional fixed mortgage` (69,919) + `Conventional adjustable mortgage (ARM)` (25,090) + already-current `Conventional home mortgage` (8,065) all become one label.
> * **Both generations are covered:** a legacy row (`Debt collection / Credit card`, 28,253) and a current row (`Debt collection / Credit card debt`, 4,541) end up with the same standardized value.
> * **Exceptions behave as designed:** `Prepaid card / Transit card` and `Cashing a check without an account` keep their original product.
> * ⚠️ One gap is visible: `Other financial service / Traveler’s/Cashier’s checks` (88 rows) gets the right *sub-product* label but its product stays `Other financial service` (Part C, note 4).

---
## Step 9 — `issue_standardized`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Issue` → `issue` (renamed in Step 21) + `issue_standardized` (derived) |
| **Type** | string → string |
| **Nulls** | 0 (0.00 %) · 166 distinct labels |
| **Operations** | profile · standardize |

**🎯 Purpose** With the 2017 taxonomy refresh the CFPB reworded many issues from the consumer's voice (*"my"*, *"I"*) to a neutral second person (*"your"*, *"you"*) and fixed punctuation. Both wordings coexist in the data.
**🧠 Logic** 14 explicit rules, some conditional on the product, because the same legacy text can belong to different current labels.

### 9.1 Distinct issues

In [ ]:
display(df_silver_valid.select('Issue').distinct().sort('Issue'))   # unique values, A -> Z

Issue
APR or interest rate
"Account opening, closing, or management"
Account terms and changes
Adding money
Advertising
Advertising and marketing
"Advertising and marketing, including promotional offers"
"Advertising, marketing or disclosures"
Application processing delay
"Application, originator, mortgage broker"


### 9.2 Issue frequencies

In [ ]:
display(df_silver_valid.groupBy("Issue").count().orderBy(F.col("Issue").desc()))   # rows per issue

Issue,count
Wrong amount charged or received,324
Written notification about debt,6599
"Was approved for a loan, but didn't receive the money",24
"Was approved for a loan, but didn't receive money",4
Vehicle was repossessed or sold the vehicle,29
Vehicle was damaged or destroyed the vehicle,7
Using a debit or ATM card,6116
Unsolicited issuance of credit card,1832
Unexpected/Other fees,103
Unexpected or other fees,232


> 💡 **What the output tells us** 166 distinct issues. Pairs such as `Dealing with my lender or servicer` / `Dealing with your lender or servicer` and `Problems caused by my funds being low` / `Problem caused by your funds being low` show the wording split that Step 9.3 resolves.

### 9.3 Standardize

In [ ]:
# "issue" resolves to the column "Issue" (case-insensitive); "product" is the trimmed RAW product.
df_silver_valid = df_silver_valid.withColumn(
    "issue_standardized",

   # Rules 1-2 - punctuation / spacing differences (no product condition needed)
   F.when(
        F.col("issue") == "Customer service / Customer relations",
        "Customer service/Customer relations"
     ) 
     .when(
        F.col("issue") == "Overdraft, savings or rewards features",
        "Overdraft, savings, or rewards features"
     )

     # Rule 3 - overdraft wording for the card product
     .when(
        (F.col("issue") == "Problem with an overdraft") & 
        (F.col("product") == "Credit card or prepaid card"),
        "Problem with overdraft"
     )
     # Rules 4-5 - payday-loan wording: first person -> second person
     # (scoped to product "Payday loan" only - see Part C note 5)
     .when(
        (F.col("issue") == "Charged fees or interest I didn't expect") &
        (F.col("product") == "Payday loan"),
        "Charged fees or interest you didn't expect"
     )
     .when(
        (F.col("issue") == "Received a loan I didn't apply for") & 
        (F.col("product") == "Payday loan"),
        "Received a loan you didn't apply for"
     )

     # Rule 6 - student loan wording
     .when(
        (F.col("issue") == "Dealing with my lender or servicer") &
        (F.col("product") == "Student loan"),
        "Dealing with your lender or servicer"
     )

     # Rules 7-10 - credit reporting wording
     .when(
        (F.col("issue") == "Incorrect information on credit report") &
        (F.col("product") == "Credit reporting"),
        "Incorrect information on your report"
     )

     .when(
        (F.col("issue") == "Improper use of my credit report") &
        (F.col("product") == "Credit reporting"),
        "Improper use of your report"
     )
     .when(
        (F.col("issue") == "Unable to get credit report/credit score") &
        (F.col("product") == "Credit reporting"),
        "Unable to get your credit report or credit score"
    )

    .when(
        (F.col("issue") == "Credit reporting company's investigation") &
        (F.col("product") == "Credit reporting"),
        "Problem with a credit reporting company's investigation into an existing problem"
    )

    # Rule 11 - bank account wording
    .when(
        (F.col("issue") == "Problems caused by my funds being low") &
        (F.col("product") == "Bank account or service"),
        "Problem caused by your funds being low"
    )

    # Rule 12 - debt collection: expand the abbreviation "Cont'd"
    .when(
        (F.col("issue") == "Cont'd attempts collect debt not owed") &
        (F.col("product") == "Debt collection"),
        "Attempts to collect debt not owed"
    )
    # Rules 13-14 - money transfers / virtual currency: "issues" -> "problem"
    .when(
        (F.col("issue") == "Other service issues") &
        (F.col("product").isin(
            "Money transfers",
            "Virtual currency"
        )),
        "Other service problem"
    )

    .when(
        (F.col("issue") == "Other transaction issues") &
        (F.col("product").isin(
            "Money transfers",
            "Virtual currency"
        )),
        "Other transaction problem"
    )
    
    # Fallback - keep the existing wording
    .otherwise(F.col("issue"))   
)

display(df_silver_valid)

product,Issue,issue_standardized
Credit reporting,Incorrect information on credit report,Incorrect information on your report
Credit reporting,Credit reporting company's investigation,Problem with a credit reporting company's investigation into an existing problem
Debt collection,Cont'd attempts collect debt not owed,Attempts to collect debt not owed
Bank account or service,Problems caused by my funds being low,Problem caused by your funds being low
Student loan,Dealing with my lender or servicer,Dealing with your lender or servicer
Credit reporting,Improper use of my credit report,Improper use of your report
Credit reporting,Unable to get credit report/credit score,Unable to get your credit report or credit score
Money transfers,Other transaction issues,Other transaction problem


> 💡 **What the output tells us** Legacy wording is rewritten to the current wording (`Incorrect information on credit report` → `Incorrect information on your report`). The original `issue` is kept next to it for traceability.

---
## Step 10 — `sub_issue` / `sub_issue_standardized`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Sub-issue` → `sub_issue` (trimmed) + `sub_issue_standardized` (derived) |
| **Type** | string → string |
| **Nulls** | 477,597 in bronze (52.83 %) — many issues simply have no sub-issue |
| **Operations** | profile · rename · trim · standardize (39 rules) |

**🎯 Purpose / 🧠 Logic** Same wording refresh as issues, scoped to three products — **Credit reporting** (9 rules), **Debt collection** (20 rules) and **Student loan** (10 rules).

### 10.1 Profile

In [ ]:
sub_issue_profile = (
    df_silver_valid.groupBy("Sub-issue")      # one row per sub-issue label
                   .count()                   # rows per label
                   .orderBy(F.col("Sub-issue").desc())
)
display(sub_issue_profile)

Sub-issue,count
"You told them to stop contacting you, but they keep trying",1056
You never received your bill or did not know a payment was due,288
"Was signed up for overdraft on card, but don't want to be",1
Was not notified of investigation status or results,1088
Used obscene/profane/abusive language,1817
"Used obscene, profane, or other abusive language",294
Unexpected increase in interest rate,165
Unable to receive car title or other problem after the loan is paid off,191
Unable to open an account,289
Trouble with how payments are handled,5621


### 10.2 Rename and trim

In [ ]:
# Chain: rename the column, then trim it, in one statement.
df_silver_valid = df_silver_valid.withColumnRenamed("Sub-issue","sub_issue").withColumn("sub_issue",F.trim(F.col("sub_issue")))
display(df_silver_valid)

product,Issue,sub_issue,complaint_id
Mortgage,"Loan modification,collection,foreclosure",null,759217
Credit reporting,Incorrect information on credit report,Account status,2141773
Consumer Loan,Managing the loan or lease,null,2163100
Credit card,Bankruptcy,null,885638
Debt collection,Communication tactics,Frequent or repeated calls,1027760


### 10.3 Standardize

In [ ]:
# Every rule is scoped by product, because the same legacy sub-issue text can exist under several products.
df_silver_valid = df_silver_valid.withColumn(
    "sub_issue_standardized",

    # CREDIT REPORTING (9 rules): "Account status" -> "Account status incorrect", "Information is not mine" -> "Information belongs to someone else" ...
    F.when(
            (F.col("sub_issue") == "Account status") &
            (F.col("product") == "Credit reporting"),
            "Account status incorrect"
        )

    .when(
        (F.col("sub_issue") == "Information is not mine") &
        (F.col("product") == "Credit reporting"),
        "Information belongs to someone else"
    )
    .when(
        (F.col("sub_issue") == "Personal information") &
        (F.col("product") == "Credit reporting"),
        "Personal information incorrect"
    )
    .when(
        (F.col("sub_issue") == "Public record") &
        (F.col("product") == "Credit reporting"),
        "Public record information inaccurate"
    )

    .when(
        (F.col("sub_issue") == "Problem getting my free annual report") &
        (F.col("product") == "Credit reporting"),
        "Problem getting your free annual credit report"
    )

    .when(
        (F.col("sub_issue") == "No notice of investigation status/result") &
        (F.col("product") == "Credit reporting"),
        "Was not notified of investigation status or results"
    )

    .when(
        (F.col("sub_issue") == "Report shared with employer w/o consent") &
        (F.col("product") == "Credit reporting"),
        "Report provided to employer without your written authorization"
    )

    .when(
        (F.col("sub_issue") == "Received marketing offer after opted out") &
        (F.col("product") == "Credit reporting"),
        "Received unsolicited financial product or insurance offers after opting out"
    )

    .when(
        (F.col("sub_issue") == "Receiving unwanted marketing/advertising") &
        (F.col("product") == "Credit reporting"),
        "Received unwanted marketing or advertising"
    )

    # DEBT COLLECTION (20 rules): first person -> second person, abbreviations expanded ("w/o" -> "without", "info" -> "information")
    .when(
        (F.col("sub_issue") == "Debt is not mine") &
        (F.col("product") == "Debt collection"),
        "Debt is not yours"
    )

    .when(
        (F.col("sub_issue") == "Debt resulted from identity theft") &
        (F.col("product") == "Debt collection"),
        "Debt was result of identity theft"
    )

    .when(
        (F.col("sub_issue") == "Debt was discharged in bankruptcy") &
        (F.col("product") == "Debt collection"),
        "Debt was already discharged in bankruptcy and is no longer owed"
    )

    .when(
        (F.col("sub_issue") == "Attempted to/Collected exempt funds") &
        (F.col("product") == "Debt collection"),
        "Collected or attempted to collect exempt funds"
    )

     .when(
        (F.col("sub_issue") == "Not given enough info to verify debt") &
        (F.col("product") == "Debt collection"),
        "Didn't receive enough information to verify debt"
    )

    .when(
        (F.col("sub_issue") == "Right to dispute notice not received") &
        (F.col("product") == "Debt collection"),
        "Didn't receive notice of right to dispute"
    )

    .when(
        (F.col("sub_issue") == "Not disclosed as an attempt to collect") &
        (F.col("product") == "Debt collection"),
        "Notification didn't disclose it was an attempt to collect a debt"
    )

    .when(
        (F.col("sub_issue") == "Contacted me after I asked not to") &
        (F.col("product") == "Debt collection"),
        "Contacted you after you asked them to stop"
    )

     .when(
        (F.col("sub_issue") == "Contacted me instead of my attorney") &
        (F.col("product") == "Debt collection"),
        "Contacted you instead of your attorney"
    )

    .when(
        (F.col("sub_issue") == "Talked to a third party about my debt") &
        (F.col("product") == "Debt collection"),
        "Talked to a third-party about your debt"
    )

    .when(
        (F.col("sub_issue") == "Impersonated an attorney or official") &
        (F.col("product") == "Debt collection"),
        "Impersonated attorney, law enforcement, or government official"
    )

    .when(
        (F.col("sub_issue") == "Indicated committed crime not paying") &
        (F.col("product") == "Debt collection"),
        "Indicated you were committing crime by not paying debt"
    )
    .when(
        (F.col("sub_issue") == "Indicated shouldn't respond to lawsuit") &
        (F.col("product") == "Debt collection"),
        "Told you not to respond to a lawsuit they filed against you"
    )

    .when(
        (F.col("sub_issue") == "Seized/Attempted to seize property") &
        (F.col("product") == "Debt collection"),
        "Seized or attempted to seize your property"
    )

    .when(
        (F.col("sub_issue") == "Sued w/o proper notification of suit") &
        (F.col("product") == "Debt collection"),
        "Sued you without properly notifying you of lawsuit"
    )

    .when(
        (F.col("sub_issue") == "Sued where didn't live/sign for debt") &
        (F.col("product") == "Debt collection"),
        "Sued you in a state where you do not live or did not sign for the debt"
    )

    .when(
        (F.col("sub_issue") == "Threatened arrest/jail if do not pay") &
        (F.col("product") == "Debt collection"),
        "Threatened to arrest you or take you to jail if you do not pay"
    )

    .when(
        (F.col("sub_issue") == "Threatened to sue on too old debt") &
        (F.col("product") == "Debt collection"),
        "Threatened to sue you for very old debt"
    )

    .when(
        (F.col("sub_issue") == "Called outside of 8am-9pm") &
        (F.col("product") == "Debt collection"),
        "Called before 8am or after 9pm"
    )

    .when(
        (F.col("sub_issue") == "Used obscene/profane/abusive language") &
        (F.col("product") == "Debt collection"),
        "Used obscene, profane, or other abusive language"
    )

    # STUDENT LOAN (10 rules): "my" -> "your", plain-language rewording
    .when(
        (F.col("sub_issue") == "Can't decrease my monthly payments") &
        (F.col("product") == "Student loan"),
        "Problem lowering your monthly payments"
    )

    .when(
        (F.col("sub_issue") == "Can't get flexible payment options") &
        (F.col("product") == "Student loan"),
        "Can't get other flexible options for repaying your loan"
    )

    .when(
        (F.col("sub_issue") == "Can't temporarily postpone payments") &
        (F.col("product") == "Student loan"),
        "Can't temporarily delay making payments"
    )

    .when(
        (F.col("sub_issue") == "Don't agree with fees charged") &
        (F.col("product") == "Student loan"),
        "Don't agree with the fees charged"
    )

    .when(
        (F.col("sub_issue") == "Having problems with customer service") &
        (F.col("product") == "Student loan"),
        "Problem with customer service"
    )   

    .when(
        (F.col("sub_issue") == "Keep getting calls about my loan") &
        (F.col("product") == "Student loan"),
        "Keep getting calls about your loan"
    )

    .when(
        (F.col("sub_issue") == "Need information about my balance/terms") &
        (F.col("product") == "Student loan"),
        "Need information about your loan balance or loan terms"
    )

    .when(
        (F.col("sub_issue") == "Qualify for a better loan than offered") &
        (F.col("product") == "Student loan"),
        "Qualified for a better loan than the one offered"
    )

    .when(
        (F.col("sub_issue") == "Received bad information about my loan") &
        (F.col("product") == "Student loan"),
        "Received bad information about your loan"
    )

    .when(
        (F.col("sub_issue") == "Trouble with how payments are handled") &
        (F.col("product") == "Student loan"),
        "Trouble with how payments are being handled"
    )
    # Fallback - keep the existing value (including NULL)
    .otherwise(F.col("sub_issue"))
)

display(df_silver_valid)

product,sub_issue,sub_issue_standardized
Credit reporting,Account status,Account status incorrect
Debt collection,Not given enough info to verify debt,Didn't receive enough information to verify debt
Debt collection,Talked to a third party about my debt,Talked to a third-party about your debt
Credit reporting,Information is not mine,Information belongs to someone else
Debt collection,Right to dispute notice not received,Didn't receive notice of right to dispute
Debt collection,Sued w/o proper notification of suit,Sued you without properly notifying you of lawsuit
Student loan,Don't agree with fees charged,Don't agree with the fees charged
Debt collection,Debt is not mine,Debt is not yours


### 10.4 Audit the rules that fired
**🧠 Logic** Show only rows where the value *changed* (`!=`) and count them per mapping.

In [ ]:
display(
    df_silver_valid
    .filter(
        F.col("sub_issue") != F.col("sub_issue_standardized")    # only rows that were rewritten
    )
    .groupBy(
        "product",
        "sub_issue",
        "sub_issue_standardized"
    )
    .count()
    .orderBy(
        "product",
        "sub_issue"
    )
)

product,sub_issue,sub_issue_standardized,count
Credit reporting,Account status,Account status incorrect,36720
Credit reporting,Information is not mine,Information belongs to someone else,32064
Credit reporting,No notice of investigation status/result,Was not notified of investigation status or results,6315
Credit reporting,Personal information,Personal information incorrect,7442
Credit reporting,Problem getting my free annual report,Problem getting your free annual credit report,6322
Credit reporting,Public record,Public record information inaccurate,8742
Credit reporting,Received marketing offer after opted out,Received unsolicited financial product or insurance offers after opting out,169
Credit reporting,Receiving unwanted marketing/advertising,Received unwanted marketing or advertising,220
Credit reporting,Report shared with employer w/o consent,Report provided to employer without your written authorization,164
Debt collection,Attempted to/Collected exempt funds,Collected or attempted to collect exempt funds,900


> 💡 **What the output tells us** **All 39 rules fired** (39 distinct mappings appear). The largest: `Debt is not mine` → `Debt is not yours` (36,325 rows), `Account status` → `Account status incorrect` (36,720), `Information is not mine` → `Information belongs to someone else` (32,064). Rows whose sub-issue was already in the current wording — or `NULL` — are untouched.

---
## Step 11 — `company` / `company_clean`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Company` → `company` (as received) + `company_clean` (derived) |
| **Type** | string → string |
| **Nulls** | 0 (0.00 %) · ≈ 4,500 distinct raw spellings |
| **Operations** | profile · diagnose hidden characters · clean |

**🎯 Purpose** The same company can be spelled slightly differently because of *invisible* characters, which splits its complaints across several groups in any `GROUP BY`.
**🧠 Logic** First *make the invisible visible* to find out what is wrong, then clean only what the diagnosis proves.

### 11.1 Profile company names

In [ ]:
display(
    df_silver_valid
    .groupBy("Company")                      # one row per spelling
    .count()                                 # complaints per spelling
    .orderBy(F.col("Company").desc())        # Z -> A
)

Company,count
"Lippman Recupero, LLC",5
reekside Recovery And Capital Management,24
"loanDepot.com, LLC",485
iServe Trust,9
"iQuantified Management Services, LLC",6
iFreedom Direct Corporation,16


> 💡 **What the output tells us** Thousands of names, so eyeballing is impossible. Some entries carry characters you cannot see (the first one begins with a non-breaking space). A targeted diagnostic follows.

### 11.2 Diagnose hidden characters
**🧠 Logic** Keep only names that contain a non-breaking space (`\u00A0`), a stray encoding character (`\u00C2`), a curly apostrophe (`\u2019`), two or more consecutive spaces, or leading/trailing spaces. Then replace each suspicious character with a visible tag (`[NBSP]`, `[SPACE]`, `[CURLY]`) to *see* what is there.

In [ ]:
# [NBSP] will show nonbreaking spaces.
# [SPACE] will show normal spaces, including repeated spaces.
# [U+00C2] will show an extra encoding character, if present.
# [CURLY] will show curly apostrophes.


display(
    df_silver_valid
    .filter(                                                    # keep only suspicious names
        F.col("company").contains("\u00A0") |                   # contains a non-breaking space
        F.col("company").contains("\u00C2") |                   # contains a stray encoding character
        F.col("company").contains("\u2019") |                   # contains a curly apostrophe
        F.col("company").rlike(r"\s{2,}") |                     # two or more whitespace characters in a row
        (F.col("company") != F.trim(F.col("company")))          # leading or trailing spaces
    )
    .select(
        "company",
        # Four nested replacements turn the invisible characters into visible tags.
        F.regexp_replace(
            F.regexp_replace(
                F.regexp_replace(
                    F.regexp_replace(
                        F.col("company"),
                        "\u00A0",
                        "[NBSP]"
                    ),
                    " ",
                    "[SPACE]"
                ),
                "\u00C2",
                "[U+00C2]"
            ),
            "\u2019",
            "[CURLY]"
        ).alias("characters_made_visible")
    )
    .distinct()                                                 # one row per unique name
    .orderBy("company")
)

company,characters_made_visible
"Advance Paychecks, LLC","Advance[SPACE]Paychecks,[SPACE][SPACE]LLC"
"American Mortgage Services, Inc","American[SPACE]Mortgage[SPACE]Services,[SPACE][SPACE]Inc"
"America’s Financial Choice, Inc.","America[CURLY]s[SPACE]Financial[SPACE]Choice,[SPACE]Inc."
Barton Creek Auto Credit LLC,Barton[SPACE]Creek[SPACE]Auto[SPACE]Credit[SPACE][SPACE]LLC
"Braviant Holdings, LLC","Braviant[NBSP]Holdings,[NBSP]LLC"
Brown’s New Credit Bureau Inc.,Brown[CURLY]s[SPACE]New[SPACE]Credit[SPACE]Bureau[SPACE]Inc.
"Concorde Auto Management, LLC DBA Concorde Auto Acceptance","Concorde[SPACE]Auto[SPACE]Management,[SPACE]LLC[SPACE][SPACE]DBA[SPACE][SPACE]Concorde[SPACE]Auto[SPACE]Acceptance"
FINANCIAL MANAGEMENT SOLUTIONS (NY),FINANCIAL[SPACE]MANAGEMENT[SPACE][SPACE]SOLUTIONS[SPACE](NY)
Franklin Financial Corporation,Franklin[SPACE]Financial[SPACE][SPACE]Corporation
"GSI Recovery, LLC","GSI[SPACE][SPACE]Recovery,[SPACE]LLC"


> 💡 **What the output tells us** Only **23 distinct names** are affected, by three kinds of defect: **non-breaking spaces** (`Braviant[NBSP]Holdings,[NBSP]LLC`), **double spaces** (`GSI[SPACE][SPACE]Recovery`) and **curly apostrophes** (`America[CURLY]s Financial Choice`). No `\u00C2` encoding damage was found. A small number — but each defect would silently split a company into two groups.

### 11.3 Clean

In [ ]:
# Rename Company -> company (snake_case). The raw spelling is kept in this column.
df_silver_valid = df_silver_valid.withColumnRenamed("Company", "company")

# company_clean starts as a copy of company with non-breaking spaces turned into normal spaces.
df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company"), "\u00A0", " ")
)

# \s+ = one or more whitespace characters -> collapse each run into a single space.
df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company_clean"), r"\s+", " ")
)

# Remove leading / trailing spaces left over.
df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.trim(F.col("company_clean"))
)

# Curly apostrophe (U+2019) -> straight apostrophe so spellings match.
df_silver_valid = df_silver_valid.withColumn(
    "company_clean",
    F.regexp_replace(F.col("company_clean"), "\u2019", "'")
)

# Show only the names that actually changed (raw vs clean).
display(
    df_silver_valid
    .filter(F.col("company") != F.col("company_clean"))
    .select("company", "company_clean")
    .distinct()
)

company,company_clean
Meridian Title Corporation,Meridian Title Corporation
"America’s Financial Choice, Inc.","America's Financial Choice, Inc."
"Concorde Auto Management, LLC DBA Concorde Auto Acceptance","Concorde Auto Management, LLC DBA Concorde Auto Acceptance"
"GSI Recovery, LLC","GSI Recovery, LLC"
"Braviant Holdings, LLC","Braviant Holdings, LLC"
"LoanNow, LLC (Closed)","LoanNow, LLC (Closed)"
"Lippman Recupero, LLC","Lippman Recupero, LLC"
STEPHEN D LANDAU,STEPHEN D LANDAU
"Kwest Mortgage Group, LLC","Kwest Mortgage Group, LLC"
Zeidman’s Jewelry & Loan of Michigan,Zeidman's Jewelry & Loan of Michigan


> 💡 **What the output tells us** Exactly the 23 diagnosed names are corrected (`Meridian Title  Corporation` → `Meridian Title Corporation`; `America’s Financial Choice, Inc.` → `America's Financial Choice, Inc.`). All other names are untouched. Use **`company_clean`** for grouping and joins; `company` preserves the source spelling. Note that upper/lower-case variants of a name (e.g. `Bank of America` vs `BANK OF AMERICA`) are *not* merged here.

---
## Step 12 — `state`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `State` → `state` |
| **Type** | string → string (2-letter code) |
| **Nulls** | 9,225 in bronze (1.02 %) → 9,214 in the valid set |
| **Operations** | profile · rename · trim · upper-case · blank → NULL |

**🎯 Purpose** Guarantee one spelling per state so that group-bys and joins to geography tables are reliable.

### 12.1 Profile

In [ ]:
display(
    df_silver_valid
    .groupBy("State")        # rows per state code
    .count()
    .orderBy("State")        # A -> Z, NULL first
)

count,State
9214,null
15,AA
371,AE
996,AK
9768,AL
258,AP
3994,AR
25,AS
19857,AZ
127160,CA


> 💡 **What the output tells us** 62 distinct codes plus `NULL` (9,214 rows). The codes are **all valid USPS abbreviations**: 50 states + DC, five territories (PR, VI, GU, AS, MP), three freely-associated states (FM, MH, PW) and the three military regions (`AA`, `AE`, `AP`). Nothing needs correcting beyond case/space safety.

### 12.2 Normalize

In [ ]:
# Rename State -> state.
df_silver_valid = df_silver_valid.withColumnRenamed("State", "state")

# Trim spaces, then force UPPER CASE ("ca " / "Ca" -> "CA").
df_silver_valid = df_silver_valid.withColumn(
    "state", F.upper(F.trim(F.col("state")))
)

# Defensive rule: an empty string "" becomes a real NULL.
df_silver_valid = df_silver_valid.withColumn(
    "state",
    F.when(F.col("state") == "", F.lit(None)).otherwise(F.col("state"))
)

# Re-profile to prove nothing was lost.
display(df_silver_valid.groupBy("state").count().orderBy("state"))

count,state
9214,null
15,AA
371,AE
996,AK
9768,AL
258,AP
3994,AR
25,AS
19857,AZ
127160,CA


> 💡 **What the output tells us** The table is **identical** to 12.1 (same 62 codes, same 9,214 NULLs): the incoming data was already clean, so the rules act as a *guard* for future loads. The blank-to-NULL rule changes nothing today because Spark's CSV reader had already turned empty fields into `NULL`.

---
## Step 13 — `zip_code`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `ZIP code` → `zip_code` |
| **Type** | string → string (`NNNXX` pattern or `NULL`) |
| **Nulls** | 9,278 in bronze (1.03 %) → 9,814 in silver (1.09 %) after invalid values are nulled |
| **Operations** | repair leading zeros · null invalid values · mask to 3 digits |

**🎯 Purpose** Produce one consistent, privacy-safe ZIP format. Two problems exist in the raw data:
1. **Lost leading zeros** — ZIPs were stored as numbers, so `01001` became `1001`. This hits New England, New Jersey, Puerto Rico and military (AE) addresses.
2. **Mixed formats and junk** — some values are 5 digits, some are already masked (`352XX`), and a few are garbage (`(1352`, `*0104`).

**🧠 Logic — a funnel, one pattern at a time**

| Stage | Pattern | Action |
|--:|---|---|
| 1 | 4 digits (`1001`) | Pad to 5 digits **only if** the state/prefix pair is plausible; else NULL |
| 2 | 1–3 digits (`707`) | Same test; else NULL |
| 3 | Anything not 5 digits and not `NNNXX` | NULL |
| 4 | 5 digits (`48382`) | Mask to `483XX` |
| — | already `NNNXX` | keep |

**Why a state check?** A padded ZIP is accepted only if its 3-digit prefix really belongs to a state that *has* leading-zero ZIPs. This avoids inventing a wrong ZIP (e.g. `CA 1040` can never be a real zero-prefixed ZIP).

| State | Accepted prefixes | | State | Accepted prefixes |
|---|---|---|---|---|
| PR | 006, 007, 009 | | NH | 030–038 |
| VI | 008 | | ME | 039–049 |
| MA | 010–027, 055 | | VT | 050–054, 056–059 |
| RI | 028–029 | | NJ | 070–089 |
| CT | 060–069 *(except 06390)* | | NY | 005 and 06390 (Fishers Island) |
| AE | 090–098 | | | |

### 13.1 Profile the raw values

In [ ]:
display(df_silver_valid.groupBy("ZIP code").count().orderBy("ZIP code"))

count,ZIP code
9267,null
1,(1352
1,*0104
1,*1212
1,-1631
1,-2914
1,0 7XX
102,006XX


> 💡 **What the output tells us** The first row is `NULL` (9,267 rows). Directly after it come **malformed values** such as `(1352`, `*0104`, `*1212` — typing noise that no repair can rescue.

### 13.2 Look at the 4-digit values

In [ ]:
four_digit_zip = (
    df_silver_valid
    .filter(F.trim(F.col("ZIP code").cast("string")).rlike(r"^\d{4}$"))   # ^ start, \d{4} exactly 4 digits, $ end
    .groupBy("state", "ZIP code")
    .count()
    .orderBy("state", "ZIP code")
)
display(four_digit_zip)

state,ZIP code,count
AE,9009,3
AE,9011,2
AE,9012,21
AE,9021,3
AE,9034,1
AE,9046,3
AE,9049,2
AE,9053,1


> 💡 **What the output tells us** The 4-digit ZIPs are dominated by Massachusetts and Connecticut, plus military `AE` addresses (e.g. `AE 9012`, 21 rows) — exactly the leading-zero victims (`09012`). A few other states appear in small numbers; those are suspicious. 🔎 *Re-verified:* there are **55,681** 4-digit ZIPs in the valid set.

### 13.3 Rename and trim

In [ ]:
# Baseline row count (kept for reference; not used further).
location_rows_before = df_silver_valid.count()

# Rename "ZIP code" -> zip_code.
df_silver_valid = df_silver_valid.withColumnRenamed("ZIP code", "zip_code")

# Cast to string (it already is) and trim spaces so the pattern tests below are reliable.
df_silver_valid = df_silver_valid.withColumn(
    "zip_code", F.trim(F.col("zip_code").cast("string"))
)
display(df_silver_valid)

state,zip_code,complaint_id
MI,48382,759217
AL,352XX,2141773
PA,177XX,2163100
ID,83854,885638
VA,23233,1027760


### 13.4 Define the leading-zero rule and repair 4-digit ZIPs
**🧠 Logic** Three reusable Spark *expressions* are defined (they compute nothing until used inside `withColumn`):
`zip_candidate` = the value padded to 5 digits · `zip_prefix` = its first three digits as a number · `valid_leading_zero` = the state/prefix test from the table above.

In [ ]:
# zip_candidate: pad with zeros on the left to 5 characters  ("1001" -> "01001")
zip_candidate = F.lpad(F.col("zip_code"), 5, "0") 

# zip_prefix: first 3 digits of the candidate as an integer ("01001" -> "010" -> 10); only for values with 1-4 digits, otherwise NULL
zip_prefix = F.when(
    # We are checking whether the original value contains one to four digits.
    F.col("zip_code").rlike(r"^\d{1,4}$"),

    # We are taking the first three characters of the candidate.
    # For 01001, we will get 010, which we are converting to integer 10 for comparison.
    # The full candidate will remain 01001.
    F.substring(zip_candidate, 1, 3).cast("int")
)

# valid_leading_zero: TRUE when the state and the prefix belong together (the table in Step 13).
# | = OR across states, & = AND within a state; between(a, b) is inclusive.
valid_leading_zero = (

    # For Puerto Rico, we will allow prefixes 006, 007 and 009.
    ((F.col("state") == "PR") & zip_prefix.isin(6, 7, 9)) |

    # For the Virgin Islands, we will allow prefix 008.
    ((F.col("state") == "VI") & (zip_prefix == 8)) |

    # We are checking for Massachusetts.
    ((F.col("state") == "MA") &
     # We will allow prefixes 010 through 027, or 055.
     (zip_prefix.between(10, 27) | (zip_prefix == 55))) |

    # For Rhode Island, we will allow prefixes 028 through 029.
    ((F.col("state") == "RI") & zip_prefix.between(28, 29)) |

    # For New Hampshire, we will allow prefixes 030 through 038.
    ((F.col("state") == "NH") & zip_prefix.between(30, 38)) |

    # For Maine, we will allow prefixes 039 through 049.
    ((F.col("state") == "ME") & zip_prefix.between(39, 49)) |

    # We are checking for Vermont.
    ((F.col("state") == "VT") &
     # We will allow prefixes 050 through 054, or 056 through 059.
     (zip_prefix.between(50, 54) | zip_prefix.between(56, 59))) |

    # For Connecticut, we are checking prefixes 060 through 069.
    ((F.col("state") == "CT") & zip_prefix.between(60, 69) &
     # We are excluding 06390; we will handle it under New York.
     (zip_candidate != "06390")) |

    # We are checking for New York.
    ((F.col("state") == "NY") &
     # We will allow prefix 005 or the specific candidate 06390.
     ((zip_prefix == 5) | (zip_candidate == "06390"))) |

    # For New Jersey, we will allow prefixes 070 through 089.
    ((F.col("state") == "NJ") & zip_prefix.between(70, 89)) |

    # For the AE military mailing region, we will allow prefixes 090 through 098.
    ((F.col("state") == "AE") & zip_prefix.between(90, 98))
)

# Apply the repair: 4-digit AND plausible -> use the padded candidate, otherwise leave the value as it is for the next stages.
df_silver_valid = df_silver_valid.withColumn(
    # We are updating the existing ZIP column.
    "zip_code",

    # We will use the padded candidate only when both checks pass.
    F.when(
        # We are checking for exactly four digits at this stage.
        F.col("zip_code").rlike(r"^\d{4}$") &

        # We are also checking whether the candidate passes our State-and-prefix rule.
        valid_leading_zero,

        # When both checks pass, we will use the candidate.
        # For example, MA with 1001 will become MA with 01001.
        zip_candidate

    # Otherwise, we will keep the current value.
    # We will handle shorter ZIPs and unresolved values in the following cells.
    ).otherwise(F.col("zip_code"))
)

> 💡 **What the output tells us** This cell has no printed output — it *changes* `zip_code`. The next cell audits what remains.

### 13.5 Which 4-digit values could not be repaired?

In [ ]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").rlike(r"^\d{4}$")     # still 4 digits = repair was refused
    )
    .groupBy(
        "state",
        "zip_code"
    )
    .count()
    .orderBy(
        "state",
        "zip_code"
    )
)

state,zip_code,count
AK,2100,1
AK,3230,1
AK,9855,1
AL,1010,2
AL,2472,1
AL,8200,1
AP,6521,1
AP,9803,1
AZ,8522,1
CA,1040,1


> 💡 **What the output tells us** 375 combinations (**432 rows**) were *refused*: they sit in states that never have leading-zero ZIPs (`AK 2100`, `AL 1010`, …) — most likely truncated or mistyped. So **55,249 of 55,681 (99.2 %)** 4-digit ZIPs were successfully repaired 🔎 *re-verified*.

### 13.6 Null the unresolved 4-digit values

In [ ]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{4}$"),    # any 4-digit value still present is unresolved
        F.lit(None)                              # -> NULL (better missing than wrong)
    ).otherwise(F.col("zip_code"))
)

display(df_silver_valid)

state,zip_code,complaint_id
MI,48382,759217
AL,352XX,2141773
PA,177XX,2163100
ID,83854,885638
VA,23233,1027760


### 13.7 Repeat for 1–3 digit values — audit first

In [ ]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").rlike(r"^\d{1,3}$")    # 1 to 3 digits
    )
    .groupBy(
        "state",
        "zip_code"
    )
    .count()
    .orderBy(
        "state",
        "zip_code"
    )
)

state,zip_code,count
CA,11,1
CA,167,1
CA,602,1
CA,802,1
CA,9,1
CO,270,1
CT,511,1
CT,660,1
GA,165,1
NJ,623,1


> 💡 **What the output tells us** 1,928 rows hold 1–3 digit values, almost all in **Puerto Rico** (1,773 rows, e.g. `PR 603` → `00603`) and the **Virgin Islands** (137 rows) — both have ZIPs that start with `00`. The few in `CA`, `NJ`, `CT`, `CO`, `GA` … (18 rows) cannot be legitimate and will fail the state/prefix test.

### 13.8 Pad the plausible ones, null the rest

In [ ]:
# Reuse the SAME rule defined in 13.4: pad only when the state/prefix is plausible.
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{1,3}$") & valid_leading_zero,
        zip_candidate
    ).otherwise(F.col("zip_code"))
)

In [ ]:
# Anything with 1-3 digits that is still left failed the test -> NULL.
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",

    F.when(
        F.col("zip_code").rlike(r"^\d{1,3}$"),
        F.lit(None)
    )
    .otherwise(F.col("zip_code"))
)

display(df_silver_valid)

state,zip_code,complaint_id
MI,48382,759217
AL,352XX,2141773
PA,177XX,2163100
ID,83854,885638
VA,23233,1027760


> 💡 **What the output tells us** 🔎 *Re-verified:* of 1,928 short ZIPs, **1,904 were repaired** and **24 were nulled**.

### 13.9 Find malformed values
**🧠 Logic** After the previous stages, a valid value must be 5 digits or `NNNXX`. Anything else is noise.

In [ ]:
display(
    df_silver_valid
    .filter(
        F.col("zip_code").isNotNull() &               # has a value ...
        ~F.col("zip_code").rlike(r"^\d{5}$") &        # ... that is NOT 5 digits ...
        ~F.col("zip_code").rlike(r"^\d{3}XX$")        # ... and NOT already masked (e.g. 352XX)
    )
    .groupBy("zip_code")
    .count()
    .orderBy("zip_code")
)

zip_code,count
(1352,1
*0104,1
*1212,1
-1631,1
-2914,1
0 7XX,1
0288-,1
0475-,1
0605-,1
07.XX,1


> 💡 **What the output tells us** 78 distinct junk values (91 rows). Typical patterns: a **truncated ZIP+4** with a trailing hyphen (`0288-`, `3008-`), **stray symbols** (`(1352`, `*0104`, `9372$`), **bracketed masks** (`[29XX`), phone-number-like fragments (`202-0`) and even a spreadsheet date corruption (`20-Mar`). None can be repaired reliably.

### 13.10 Null the malformed values

In [ ]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",

    F.when(
        F.col("zip_code").isNotNull() &
        ~F.col("zip_code").rlike(r"^\d{5}$") &
        ~F.col("zip_code").rlike(r"^\d{3}XX$"),
        F.lit(None)                                    # invalid -> NULL
    )

    .otherwise(F.col("zip_code"))
)

display(df_silver_valid)

state,zip_code,complaint_id
MI,48382,759217
AL,352XX,2141773
PA,177XX,2163100
ID,83854,885638
VA,23233,1027760


### 13.11 Preview the masking, then confirm what remains
**🧠 Logic** The CFPB itself masks some ZIPs to three digits (`352XX`). To make every record the same granularity, 5-digit ZIPs are masked too. First a *dry run* with `.limit(20)` shows the effect without changing the table.

In [ ]:
display(
    df_silver_valid.select(
        "state", "zip_code",
        F.when(
            F.col("zip_code").rlike(r"^\d{5}$"),                                   # 5-digit value?
            F.concat(F.substring(F.col("zip_code"), 1, 3), F.lit("XX"))           # first 3 chars + "XX"
        ).otherwise(F.col("zip_code")).alias("zip_reporting_preview")             # else keep as is
    ).limit(20)                                                                    # only 20 rows
)

state,zip_code,zip_reporting_preview
MI,48382,483XX
AL,352XX,352XX
PA,177XX,177XX
ID,83854,838XX
VA,23233,232XX
MN,48322,483XX
CA,92111,921XX
VA,224XX,224XX


In [ ]:
# Which distinct values are NOT yet masked? Should be 5-digit codes only.
display(
    df_silver_valid
    .filter(
        F.col("zip_code").isNotNull() &
        ~F.col("zip_code").rlike(r"^\d{3}XX$")
    )
    .select("zip_code")
    .distinct()
)

zip_code
75007
29210
80305
97128
73463
60070


> 💡 **What the output tells us** `48382` → `483XX`, `83854` → `838XX`, while `352XX` is kept. The second check confirms that **only clean 5-digit ZIPs remain unmasked** — the cleaning stages worked.

### 13.12 Apply the mask

In [ ]:
df_silver_valid = df_silver_valid.withColumn(
    "zip_code",
    F.when(
        F.col("zip_code").rlike(r"^\d{5}$"),          # 5-digit ZIP ...
        F.concat(
            F.substring(F.col("zip_code"), 1, 3),     # ... keep the first 3 digits (substring is 1-based)
            F.lit("XX")                               # ... and replace the last two with XX
        )
    ).otherwise(F.col("zip_code"))                    # NULL and already-masked values pass through
)

display(df_silver_valid)

state,zip_code,complaint_id
MI,483XX,759217
AL,352XX,2141773
PA,177XX,2163100
ID,838XX,885638
VA,232XX,1027760


> 💡 **What the output tells us** `zip_code` now holds only `NNNXX` or `NULL`.

#### 📈 ZIP repair scorecard 🔎 *re-verified on the supplied CSV*
| Raw ZIP pattern | Rows | Outcome |
|---|--:|---|
| 5 digits | 593,030 | masked → `NNNXX` |
| Already masked `NNNXX` | 236,934 | kept |
| 4 digits | 55,681 | **55,249 repaired**, 432 → NULL |
| 1–3 digits | 1,928 | **1,904 repaired**, 24 → NULL |
| Malformed | 91 | → NULL |
| Already NULL | 9,267 | stays NULL |
| **Total** | **896,931** | **887,117 usable · 9,814 NULL (1.09 %)** |

> The trade-off: ZIPs are only resolved to a 3-digit area (≈ 1,000 areas nationally). That is enough for regional analysis and consistent with CFPB's own masking, but it cannot be joined to 5-digit ZIP tables.

---
## Step 14 — `tags`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Tags` → `tags` |
| **Type** | string → string |
| **Nulls** | 777,945 in bronze (86.06 %) → 771,791 in the valid set |
| **Operations** | profile · temporal check · rename · trim |

**🎯 Purpose** `Tags` flags complaints from special populations (*Older American*, *Servicemember*). Is its heavy null share a timing artefact or just "no flag"?
**🧠 Logic** Two profiles: counts per value, then the **first and last date** each value is seen. If `NULL` appears across the whole timeline, it simply means "no tag".

### 14.1 Value counts

In [ ]:
# Rows per tag value, Z -> A (NULL last).
display(df_silver_valid.groupBy("Tags").count().orderBy(F.col("Tags").desc()))

Tags,count
Servicemember,49802
"Older American, Servicemember",10002
Older American,65336
null,771791


### 14.2 When is each value observed?
`F.min / F.max` on `date_received` give the first and last date; `F.count("*")` counts rows.

In [ ]:
# Same grouping, plus the first / last date each value was received.
display(
    df_silver_valid
    .groupBy("Tags")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Tags")
)

count,first_observed,last_observed,Tags
771791,2011-12-01,2017-11-15,null
65336,2011-12-01,2017-11-13,Older American
10002,2011-12-01,2017-11-14,"Older American, Servicemember"
49802,2011-12-01,2017-11-15,Servicemember


> 💡 **What the output tells us** Four states: *Servicemember* 49,802 · *Older American* 65,336 · both 10,002 · `NULL` 771,791 (86.0 %). Every value spans **the full period** (2011-12-01 → 2017-11), so `NULL` simply means *no special-population tag* — not "missing data". Kept as `NULL`.

### 14.3 Rename and trim

In [ ]:
# Rename Tags -> tags and trim spaces in one chained statement.
df_silver_valid = (
    df_silver_valid.
    withColumnRenamed("Tags","tags")
    .withColumn("tags",F.trim(F.col("tags")))
)

display(df_silver_valid)

complaint_id,tags
759217,null
2141773,null
2163100,Older American
885638,Older American
1027760,null


---
## Step 15 — `submitted_via`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Submitted via` → `submitted_via` |
| **Type** | string → string · **Nulls** 0 · 6 distinct values |
| **Operations** | profile · temporal check · rename · trim |

**🎯 Purpose** Channel through which the complaint arrived.

In [ ]:
# Rows per channel, largest first.
display(
    df_silver_valid
    .groupBy("Submitted via")
    .count()
    .orderBy(F.col("count").desc())
)

Submitted via,count
Web,627854
Referral,142880
Phone,58504
Postal mail,55065
Fax,12277
Email,351


In [ ]:
# First / last date each channel is observed.
display(
    df_silver_valid
    .groupBy("Submitted via")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Submitted via")
)

Submitted via,count,first_observed,last_observed
Email,351,2011-12-07,2017-10-24
Fax,12277,2011-12-05,2017-11-14
Phone,58504,2011-12-01,2017-11-14
Postal mail,55065,2011-12-01,2017-11-14
Referral,142880,2011-12-01,2017-11-14
Web,627854,2011-12-01,2017-11-15


> 💡 **What the output tells us** The **web** dominates (627,854 rows, 70.0 %), followed by **referrals** from other agencies (15.9 %), phone (6.5 %), postal mail (6.1 %), fax (1.4 %) and email (351 rows). All six channels are present from late 2011 (email since 2011-12-07). The values are already clean — only a rename and trim are needed.

In [ ]:
# Rename to snake_case and trim.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed("Submitted via","submitted_via")
    .withColumn("submitted_via",F.trim(F.col("submitted_via")))
)

display(df_silver_valid)

complaint_id,submitted_via
759217,Referral
2141773,Web
2163100,Web
885638,Web
1027760,Web


---
## Step 16 — `consumer_consent_provided`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Consumer consent provided?` → `consumer_consent_provided` |
| **Type** | string → string |
| **Nulls** | 528,549 in bronze (58.47 %) → 521,497 in the valid set |
| **Operations** | profile · temporal check · **hypothesis test** · rename · trim |

**🎯 Purpose** Understand *why* more than half of the rows are `NULL` before deciding what to do with them.

In [ ]:
# Rows per consent value.
display(
    df_silver_valid
    .groupBy("Consumer consent provided?")
    .count()
    .orderBy(F.col("count").desc())
)

count,Consumer consent provided?
521497,null
199971,Consent provided
161784,Consent not provided
13134,Other
545,Consent withdrawn


In [ ]:
# First / last date each value is observed.
display(
    df_silver_valid
    .groupBy("Consumer consent provided?")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Consumer consent provided?")
)

count,first_observed,last_observed,Consumer consent provided?
521497,2011-12-01,2017-11-15,null
161784,2015-03-19,2017-11-15,Consent not provided
199971,2015-03-19,2017-11-04,Consent provided
545,2015-06-27,2017-11-08,Consent withdrawn
13134,2015-03-19,2017-11-15,Other


**🧠 Hypothesis** The values first appear on **2015-03-19**, so *every complaint before that date must be NULL* — the field didn't exist. Test it:

In [ ]:
# Hypothesis test: what consent values exist BEFORE the field was introduced (2015-03-19)?
# (Spark compares the date column with the text "2015-03-19" by casting it to a date.)
display(
    df_silver_valid
    .filter(F.col("date_received") < "2015-03-19")
    .groupBy("Consumer consent provided?")
    .count()
)

count,Consumer consent provided?
362760,None


> 💡 **What the output tells us** The test confirms it: **all 362,760 complaints received before 2015-03-19 are `NULL`** — and nothing else. So 362,760 of the 521,497 NULLs are *structural* ("not collected yet"). The remaining 158,737 are post-launch complaints with no stated consent. Consent was *provided* for 199,971 complaints, which lines up with the ≈ 199,970 non-empty narratives (Step 22). `NULL` is kept — it carries meaning.

In [ ]:
# Rename and trim.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Consumer consent provided?",
        "consumer_consent_provided"
    )
    .withColumn(
        "consumer_consent_provided",
        F.trim(F.col("consumer_consent_provided"))
    )
)

display(df_silver_valid)

complaint_id,consumer_consent_provided
759217,null
2141773,Consent provided
2163100,Consent provided
885638,null
1027760,null


---
## Step 17 — `company_public_response`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Company public response` → `company_public_response` |
| **Type** | string → string |
| **Nulls** | 646,002 in bronze (71.46 %) → 638,951 in the valid set |
| **Operations** | profile · temporal check · rename · trim |

**🎯 Purpose** An *optional* public statement by the company. Understand the vocabulary and when each statement was available.

In [ ]:
# Rows per public-response text, largest first.
display(
    df_silver_valid
    .groupBy("Company public response")
    .count()
    .orderBy(F.col("count").desc())
)

count,Company public response
638951,null
147983,Company has responded to the consumer and the CFPB and chooses not to provide a public response
52472,Company chooses not to provide a public response
41190,Company believes it acted appropriately as authorized by contract or law
3861,Company believes the complaint is the result of a misunderstanding
3561,Company disputes the facts presented in the complaint
2979,Company believes complaint caused principally by actions of third party outside the control or direction of the company
2711,Company believes complaint is the result of an isolated error
1724,Company can't verify or dispute the facts in the complaint
1451,Company believes complaint represents an opportunity for improvement to better serve consumers


In [ ]:
# First / last date each statement is observed.
display(
    df_silver_valid
    .groupBy("Company public response")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Company public response")
)

count,first_observed,last_observed,Company public response
638951,2011-12-01,2017-11-15,null
2979,2014-02-01,2017-11-07,Company believes complaint caused principally by actions of third party outside the control or direction of the company
2711,2014-09-01,2017-11-13,Company believes complaint is the result of an isolated error
48,2015-03-19,2017-09-08,Company believes complaint relates to a discontinued policy or procedure
1451,2015-02-17,2017-11-09,Company believes complaint represents an opportunity for improvement to better serve consumers
41190,2013-09-12,2017-11-15,Company believes it acted appropriately as authorized by contract or law
3861,2013-06-26,2017-11-14,Company believes the complaint is the result of a misunderstanding
1724,2013-11-12,2017-11-13,Company can't verify or dispute the facts in the complaint
52472,2013-08-09,2016-03-02,Company chooses not to provide a public response
3561,2014-10-02,2017-11-14,Company disputes the facts presented in the complaint


> 💡 **What the output tells us** Ten fixed statements plus `NULL` (638,951 rows, 71.2 %). Nothing is recorded before **2012-04-14**, and the statements appear at different dates — the CFPB added options over time. Note that two statements mean almost the same thing: *"Company has responded … and chooses not to provide a public response"* (147,983) and the older *"Company chooses not to provide a public response"* (52,472; used only 2013-08-09 → 2016-03-02). They are left as published — a merge decision for the gold layer.

In [ ]:
# Rename and trim.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Company public response",
        "company_public_response"
    )
    .withColumn(
        "company_public_response",
        F.trim(F.col("company_public_response"))
    )
)

display(df_silver_valid)

complaint_id,company_public_response
759217,null
2141773,Company has responded to the consumer and the CFPB and chooses not to provide a public response
2163100,null
885638,null


---
## Step 18 — `company_response_to_consumer`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Company response to consumer` → `company_response_to_consumer` |
| **Type** | string → string · **Nulls** 0 · 8 distinct values |
| **Operations** | profile · temporal check · rename · trim |

**🎯 Purpose** How the company closed the complaint — the most important outcome field.

In [ ]:
# Rows per outcome, largest first.
display(
    df_silver_valid
    .groupBy("Company response to consumer")
    .count()
    .orderBy(F.col("count").desc())
)

Company response to consumer,count
Closed with explanation,681069
Closed with non-monetary relief,107090
Closed with monetary relief,56526
Closed without relief,17851
Closed,17471
In progress,7015
Closed with relief,5296
Untimely response,4613


In [ ]:
# First / last date each outcome is observed.
display(
    df_silver_valid
    .groupBy("Company response to consumer")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Company response to consumer")
)

Company response to consumer,count,first_observed,last_observed
Closed,17471,2011-12-02,2017-04-21
Closed with explanation,681069,2011-12-01,2017-11-15
Closed with monetary relief,56526,2011-12-06,2017-11-14
Closed with non-monetary relief,107090,2011-12-05,2017-11-14
Closed with relief,5296,2011-12-01,2012-05-29
Closed without relief,17851,2011-12-01,2012-10-15
In progress,7015,2017-05-04,2017-11-15
Untimely response,4613,2011-12-01,2017-10-31


> 💡 **What the output tells us**
> * **76 %** of complaints (681,069) were *closed with explanation*; **monetary relief** 56,526 (6.3 %); **non-monetary relief** 107,090 (11.9 %).
> * **Legacy values:** `Closed with relief` (last seen **2012-05-29**) and `Closed without relief` (last seen **2012-10-15**) were later split into *monetary / non-monetary relief*; plain `Closed` ends **2017-04-21**. They are left as published — analysts computing a "relief rate" should group them deliberately.
> * `In progress` starts **2017-05-04**: these were still open when the data was exported.

In [ ]:
# Rename and trim.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Company response to consumer",
        "company_response_to_consumer"
    )
    .withColumn(
        "company_response_to_consumer",
        F.trim(F.col("company_response_to_consumer"))
    )
)

display(df_silver_valid)

complaint_id,company_response_to_consumer
759217,Closed with explanation
2141773,Closed with explanation
2163100,Closed with explanation
885638,Closed with explanation


---
## Step 19 — `timely_response`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Timely response?` → `timely_response` |
| **Type** | string (`Yes` / `No`) · **Nulls** 0 |
| **Operations** | profile · rename · trim |

**🎯 Purpose** Did the company respond within the required time frame?

In [ ]:
# Rows per value.
display(
    df_silver_valid
    .groupBy("Timely response?")
    .count()
    .orderBy(F.col("count").desc())
)

Timely response?,count
Yes,872402
No,24529


> 💡 **What the output tells us** **97.3 %** of responses were timely (872,402 *Yes* vs 24,529 *No*). Only two clean values, so no standardization is needed.

In [ ]:
# Rename "Timely response?" (the "?" is awkward in code) and trim. No display - nothing to look at.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Timely response?",
        "timely_response"
    )
    .withColumn(
        "timely_response",
        F.trim(F.col("timely_response"))
    )
)

---
## Step 20 — `consumer_disputed`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Consumer disputed?` → `consumer_disputed` |
| **Type** | string (`Yes` / `No`) |
| **Nulls** | 135,408 (15.1 % of the valid set) |
| **Operations** | profile · temporal check · rename · trim |

**🎯 Purpose** Did the consumer dispute the company's answer? ~15 % are `NULL`; find out whether that is random.

In [ ]:
# Rows per value.
display(
    df_silver_valid
    .groupBy("Consumer disputed?")
    .count()
    .orderBy(F.col("count").desc())
)

Consumer disputed?,count
No,614652
Yes,146871
null,135408


In [ ]:
# First / last date each value is observed.
display(
    df_silver_valid
    .groupBy("Consumer disputed?")
    .agg(
        F.count("*").alias("count"),
        F.min("date_received").alias("first_observed"),
        F.max("date_received").alias("last_observed")
    )
    .orderBy("Consumer disputed?")
)

count,first_observed,last_observed,Consumer disputed?
135408,2017-04-24,2017-11-15,null
614652,2011-12-01,2017-04-22,No
146871,2011-12-01,2017-04-21,Yes


> 💡 **What the output tells us** The dates answer the question: `Yes`/`No` stop after **2017-04-22**, and every `NULL` begins on **2017-04-24** and runs to the end of the data. The CFPB **stopped collecting this field in April 2017**, so these NULLs are *structural*, not data loss. Among the 761,523 complaints that do have an answer, **146,871 (19.3 %)** were disputed. Analyses of dispute rates must stop at April 2017.

In [ ]:
# Rename and trim. No display - nothing to look at.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Consumer disputed?",
        "consumer_disputed"
    )
    .withColumn(
        "consumer_disputed",
        F.trim(F.col("consumer_disputed"))
    )
)

---
## Step 21 — `issue` (rename)

**🎯 Purpose** `Issue` was deliberately left with its original name during Steps 9 and 10 so rule conditions stayed easy to read. Now it receives its `snake_case` name; `issue_standardized` (built in Step 9) is unaffected.

In [ ]:
# Rename Issue -> issue. The values themselves are not trimmed or changed.
df_silver_valid = (
    df_silver_valid
    .withColumnRenamed(
        "Issue",
        "issue"
    )
)
display(df_silver_valid)

complaint_id,issue,issue_standardized
759217,"Loan modification,collection,foreclosure","Loan modification,collection,foreclosure"
2141773,Incorrect information on credit report,Incorrect information on your report
2163100,Managing the loan or lease,Managing the loan or lease
885638,Bankruptcy,Bankruptcy
1027760,Communication tactics,Communication tactics


---
## Step 22 — Drop `Consumer complaint narrative`

| 🏷️ Column card | |
|---|---|
| **Source → Silver** | `Consumer complaint narrative` → *(dropped)* |
| **Type** | string (free text) |
| **Nulls** | 704,013 in bronze (77.88 %) → 696,961 in the valid set |
| **Operation** | profile · drop |

**🎯 Purpose** Decide whether the heaviest column earns a place in silver.
**🧠 Logic** Count populated vs empty narratives. A column that is **78 % empty**, very large, and not needed for structured analysis (the masked text contains `XXXX` placeholders) makes the silver table slow and bulky. The text is **not lost** — it remains in bronze.

In [ ]:
display(
    df_silver_valid.select(
        F.count("*").alias("total_rows"),                        # all rows

        F.sum(                                                    # rows WITHOUT a narrative
            F.when(
                F.col("Consumer complaint narrative").isNull(),
                1
            ).otherwise(0)
        ).alias("null_count"),

        F.sum(                                                    # rows WITH a narrative
            F.when(
                F.col("Consumer complaint narrative").isNotNull(),
                1
            ).otherwise(0)
        ).alias("non_null_count")
    )
)

total_rows,null_count,non_null_count
896931,696961,199970


> 💡 **What the output tells us** Only **199,970 of 896,931 rows (22.3 %)** have a narrative. Interesting side-fact: 704,013 − 696,961 = **7,052** — every quarantined row had *no* narrative, i.e. none of the rejected rows contained consumer text.

### Drop it

In [ ]:
# .drop() removes the column from the silver DataFrame only; the bronze CSV is untouched.
df_silver_valid = df_silver_valid.drop(
    "Consumer complaint narrative"
)

display(df_silver_valid)

complaint_id,issue,company
759217,"Loan modification,collection,foreclosure",M&T BANK CORPORATION
2141773,Incorrect information on credit report,"TRANSUNION INTERMEDIATE HOLDINGS, INC."
2163100,Managing the loan or lease,"CITIZENS FINANCIAL GROUP, INC."
885638,Bankruptcy,AMERICAN EXPRESS COMPANY
1027760,Communication tactics,"CITIBANK, N.A."


> 💡 **For NLP / text-mining use cases**, re-read the narrative from bronze and join on `complaint_id` (199,970 narratives are available).

---
## Step 23 — Final validation and column order

**🎯 Purpose** Prove the silver table is sound *before* it is written, and fix the column order so the schema is predictable.

### 23.1 Is the key still unique?

In [ ]:
display(
    df_silver_valid.select(
        F.count("*").alias("total_rows"),                          # rows
        F.count("complaint_id").alias("non_null_ids"),             # non-NULL ids
        F.countDistinct("complaint_id").alias("distinct_ids")      # unique ids
    )
)

total_rows,non_null_ids,distinct_ids
896931,896931,896931


> 💡 **What the output tells us** 896,931 rows = 896,931 non-null IDs = 896,931 distinct IDs → **one row per complaint**, no duplicates.

### 23.2 Select the final 22 columns in a logical order
**🧠 Logic** `select` with an explicit list does two things: it chooses *which* columns survive (a whitelist) and sets their order — identifiers and dates first, then product/issue/company with their standardized twins, then geography, channel and outcome.

In [ ]:
df_silver_valid = df_silver_valid.select(
    # --- key and dates ---
    "complaint_id",
    "date_received",
    "date_sent_to_company",

    # --- what the complaint is about: raw label followed by its standardized twin ---
    "product",
    "product_standardized",
    "sub_product",
    "sub_product_standardized",

    "issue",
    "issue_standardized",
    "sub_issue",
    "sub_issue_standardized",

    # --- who ---
    "company",
    "company_clean",

    # --- where / who is affected ---
    "state",
    "zip_code",
    "tags",

    # --- how it was filed ---
    "submitted_via",
    "consumer_consent_provided",

    # --- outcome ---
    "company_public_response",
    "company_response_to_consumer",

    "timely_response",
    "consumer_disputed"
)

### 23.3 Final data-quality scorecard
**🧠 Logic** Re-test the rules on the finished table — key integrity, the date rule, and the ZIP format (`NNNXX` only).

In [ ]:
final_dq = df_silver_valid.select(
    F.count("*").alias("total_rows"),

    F.countDistinct("complaint_id").alias("distinct_complaint_ids"),

    # NULL keys
    F.sum(
        F.when(F.col("complaint_id").isNull(), 1).otherwise(0)
    ).alias("null_complaint_ids"),

    # Rows still violating the date-sequence rule (must be 0)
    F.sum(
        F.when(
            F.col("date_sent_to_company") < F.col("date_received"),
            1
        ).otherwise(0)
    ).alias("invalid_date_sequence"),

    # Non-NULL ZIPs that are NOT in NNNXX format (must be 0)
    F.sum(
        F.when(
            F.col("zip_code").isNotNull() &
            ~F.col("zip_code").rlike(r"^\d{3}XX$"),
            1
        ).otherwise(0)
    ).alias("invalid_zip_format")
)

display(final_dq)

total_rows,distinct_complaint_ids,null_complaint_ids,invalid_date_sequence,invalid_zip_format
896931,896931,0,0,0


> 💡 **What the output tells us** ✅ All four checks pass: 896,931 rows, 896,931 distinct IDs, and **0** NULL keys, invalid date sequences and invalid ZIP formats. The table is ready to publish.

---
## Step 24 — Publish the Silver Delta table

**🎯 Purpose** Persist the result as a managed **Delta** table so downstream notebooks, SQL endpoints and Power BI can query it.

### 24.1 Create the schema (namespace)

In [ ]:
# "IF NOT EXISTS" makes the cell safe to re-run.
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

DataFrame[]

> The empty `DataFrame[]` is the normal return value of a DDL statement.

### 24.2 Write the table

In [ ]:
df_silver_valid.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.consumer_complaints")

| Part | Meaning |
|---|---|
| `.format("delta")` | Delta Lake storage — ACID transactions, versioning / time-travel, fast reads |
| `.mode("overwrite")` | Replace the table's contents on every run → the notebook is **idempotent** (re-running gives the same result, no duplicates) |
| `.option("overwriteSchema","true")` | Allow the column layout to change between runs |
| `.saveAsTable("silver.consumer_complaints")` | Register it as a managed table named `consumer_complaints` in the `silver` schema |

> 💡 **Result:** `silver.consumer_complaints` — 896,931 rows × 22 columns.

---
## Step 25 — Quarantine table and CSV export

### 25.1 Shape the quarantine table
**🧠 Logic** The quarantine DataFrame (Step 6.2) still has capitalized source names. Select a compact set of columns and alias them to match the silver naming.

In [ ]:
df_quarantine_final = df_quarantine.select(
    "complaint_id",
    "date_received",
    "date_sent_to_company",
    F.col("Product").alias("product"),      # .alias() renames a column inside a select
    F.col("Company").alias("company"),
    F.col("State").alias("state"),
    "dq_status",                            # FAIL
    "dq_rule",                              # DATE SEQUENCE
    "dq_reason"                             # human-readable explanation
)

display(df_quarantine_final)

complaint_id,date_received,date_sent_to_company,product,company,state,dq_status,dq_rule,dq_reason
730165,2014-02-25,2014-02-24,Debt collection,"Navient Solutions, LLC.",RI,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
210633,2012-12-15,2012-12-14,Credit card,BARCLAYS BANK DELAWARE,CA,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
722527,2014-02-20,2014-02-19,Mortgage,"Stearns Ventures, LLC",CA,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
663076,2014-01-10,2014-01-09,Mortgage,"BANK OF AMERICA, NATIONAL ASSOCIATION",MD,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received
804419,2014-04-11,2014-04-10,Mortgage,OCWEN LOAN SERVICING LLC,FL,FAIL,DATE SEQUENCE,Date sent to company is earlier than date received


> 💡 **What the output tells us** Each rejected row stays traceable: who, when, which product, and exactly which rule it failed.

### 25.2 Write the quarantine table

In [ ]:
df_quarantine_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.consumer_complaints_quarantine")

### 25.3 Export a single CSV copy
**🧠 Logic** Read the Delta table back (this also proves the write succeeded), count it, and write one CSV file for users who need a flat extract.

In [ ]:
# Read the table we just saved (spark.table loads a registered table).
df_export = spark.table("silver.consumer_complaints")

# f-string: {value:,} formats the number with thousands separators.
print(f"Rows to export: {df_export.count():,}")

(
    df_export.coalesce(1)                      # merge all partitions into ONE -> a single CSV part-file
    .write.mode("overwrite")                   # replace any earlier export
    .option("header", "true")                  # first line = column names
    .csv("Files/exports/consumer_complaints_full")   # output FOLDER (Spark writes part-*.csv inside it)
)

Rows to export: 896,931


> 💡 **What the output tells us** The read-back returns **896,931** rows — the Delta write is complete and matches the scorecard in Step 23.3. `coalesce(1)` is fine at this size (the narrative is gone); for much larger tables prefer keeping several partitions.

✅ **Bronze → Silver complete.**

---
# Part C — Lead reviewer's notes

## C.1 Row and column reconciliation

| Check | Result |
|---|---|
| Bronze rows | **903,983** |
| − Quarantined (date-sequence rule) | − 7,052 |
| = Silver rows | **896,931** ✅ equals the count in Step 6.3 and the exported rows in Step 25.3 |
| Unique key | 896,931 distinct `complaint_id`, 0 NULL |
| Columns | 18 bronze → 22 silver (17 retained, 1 dropped, 5 derived) |
| Data types changed | 2 text → `date` (`date_received`, `date_sent_to_company`) |
| Rules still violated in silver | **0** date-sequence · **0** ZIP-format |

## C.2 What was done well
* **Bronze is never modified**; every step is reproducible from the raw file.
* **Nothing is silently dropped** — rule failures are quarantined with a reason, and the two filters are exact complements.
* **Raw and standardized values live side by side** (`product` / `product_standardized`, `company` / `company_clean`), so every mapping is auditable.
* **Nulls are interpreted, not blindly filled** — first/last-observed profiling proved that most NULLs are structural (consent, public response, dispute, tags).
* **Conservative ZIP repair** — a padded ZIP is accepted only if state and prefix agree; otherwise NULL is preferred over a wrong value.
* **Idempotent writes** (`overwrite`) and a final scorecard before publishing.

## C.3 Observations and recommendations
Items marked 🔎 were re-verified against the supplied CSV. None changes the results documented above; they are suggestions for the next iteration.

| # | Where | Observation | Suggested action |
|--:|---|---|---|
| 1 | Steps 4.1, 5.2 | `min` / `max` were computed on **text**, giving `1/1/2012`–`9/9/2017` (alphabetical). The real range is **2011-12-01 → 2017-11-15** 🔎 | Compute min/max *after* `to_date` |
| 2 | Step 5.2 | The null count for `Date sent to company` actually tests `Date received`. Both have 0 nulls, so the result is correct by coincidence | Change `F.col("Date received")` to `F.col("Date sent to company")` |
| 3 | Step 6 | All 7,052 quarantined rows are exactly **one day early**, dated 2011-12-02 → 2014-04-26, and none occurs later 🔎. This looks like a systematic time-stamp artefact, not random error | Keep quarantine, but consider an *auto-correct* variant (`sent = received`) with a flag, and ask the data owner about the cause |
| 4 | Step 7 | `Other financial service / Traveler’s/Cashier’s checks` (88 rows) keeps product `Other financial service`: the rule lists spellings with spaces or straight apostrophes, the data has `Traveler’s/Cashier’s checks` (curly apostrophes, no spaces). The *sub-product* is mapped correctly | Normalize `’` → `'` before the product rule, as Step 8 already does |
| 5 | Step 9 | Issue rules for *"…I didn't expect"* and *"…I didn't apply for"* are scoped to product `Payday loan`; **315** `Consumer Loan` rows (292 + 23) keep the legacy first-person wording 🔎 | Extend the rules to `Consumer Loan` (or drop the product condition) |
| 6 | Step 8.3 | Rules for `'Consumer loan'` (lower-case *l*) never match — the product is spelled `Consumer Loan` (0 rows with the lower-case form 🔎). The later upper-case rules do the real work | Remove the dead block or compare with `F.lower()` |
| 7 | Steps 17–18 | Legacy/overlapping labels remain (`Closed with relief` vs *monetary/non-monetary relief*; two "chooses not to provide a public response" texts) | Decide a grouping in the gold layer; document it |
| 8 | Step 13 | `zip_code` keeps only a 3-digit area (by design) and 9,814 rows (1.09 %) are NULL 🔎 | Fine for regional analysis; keep the 5-digit ZIP in bronze if street-level joins are ever needed |
| 9 | Step 22 | The narrative (199,970 texts) is dropped from silver | Publish it as a separate table keyed on `complaint_id` if text analytics is planned |
| 10 | Not in pipeline | 146 valid rows have a received→sent lag of **more than 365 days** (maximum 1,962) 🔎 — likely date typos | Add a plausibility rule (e.g. lag ≤ 365 days) and quarantine or flag |
| 11 | Step 2.2 | The null-profile loop runs 36 Spark jobs and `df_silver_valid` is reassigned ~40 times, each triggering a re-plan | Compute all null counts in one `select`; consider `.cache()` before the audits |
| 12 | Whole notebook | Several cached outputs in the original file were captured out of order (e.g. the preview after the date drop in Step 4.3 shows neither date column, yet later previews show `date_received`). Run top-to-bottom, the cell order documented here is consistent | Do a clean **Run all** in a fresh session before each release so saved outputs match the code |

## C.4 What comes next — Gold layer ideas
* Monthly complaint volume by `product_standardized` and `company_clean`
* Relief rate and timely-response rate per company (grouping the legacy response labels first)
* Regional heat-map by `state` / 3-digit ZIP
* Channel mix over time (`submitted_via`)
* Text analytics on the narratives joined back from bronze